# Survey-Aware Explainable Machine Learning for Four-Class Anaemia Severity Classification

## Women aged 15–49 in India — NFHS-5 final 40-column research extract

**Purpose.** This notebook implements a restart-safe, leakage-controlled and
survey-aware pipeline for contemporaneous four-class anaemia severity
classification: **No anaemia / Mild / Moderate / Severe**.

**Primary data.** `anaemia_women_research_40cols.csv` contains 724,115 women,
the NFHS complex-survey design fields, 707 state–district domains, clinically
and policy-relevant predictors, adjusted haemoglobin for sensitivity analysis,
and the raw NFHS anaemia target.

**Interpretation boundary.** The study is cross-sectional. Predictions and SHAP
values describe model associations; they do not establish causality, prognosis,
diagnosis or treatment recommendations.

**Restart policy.** Every expensive stage has a versioned manifest and an
atomic checkpoint. A stage is reused only when the source hash, feature schema,
pipeline version and stage parameters match exactly.

## v3.1 pre-paper safety protocol

This revision preserves the executed v3 development notebook as a pilot archive.
The earlier development test was exposed and is not represented as a pristine final
holdout. Publication mode therefore uses a new split seed, requires an explicit
protocol amendment, and treats external/cross-wave validation as a publication gate.

Paper execution is deliberately two-pass:

1. Set `RUN_MODE="paper"` and keep `UNLOCK_FINAL_TEST=False`; run all cells to
   complete nested and geographic validation checkpoints.
2. Confirm the readiness report, then set `UNLOCK_FINAL_TEST=True` and run all
   cells again. Compatible training checkpoints load; the locked test is evaluated.

Never change features, models, endpoints, or thresholds after the final-test unlock.


In [ ]:
# 0. Colab/Runtime setup
import os
import sys
import subprocess
import importlib.util

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "imblearn": "imbalanced-learn",
    "optuna": "optuna",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "catboost": "catboost",
    "shap": "shap",
    "pyarrow": "pyarrow",
    "pyreadstat": "pyreadstat",
    "seaborn": "seaborn",
    "joblib": "joblib",
    "tqdm": "tqdm",
}

missing_packages = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing_packages
    ])
else:
    print("All required packages are already installed.")


## 1. Run configuration

1. Upload `anaemia_women_research_40cols.csv` to Google Drive.
2. Keep `RUN_MODE="development"` for a fast end-to-end test.
3. Use `RUN_MODE="paper"` for final Optuna, nested-CV, bootstrap and geographic results.
4. To rerun only selected stages, add names to `FORCE_STAGES`, for example
   `{"splits", "lightgbm", "bootstrap"}`.

The default Colab project directory is stored in Drive, so completed Optuna
trials, fold models, probabilities, SHAP arrays and bootstrap replicates survive
runtime disconnections. Because the final feature schema differs from older
30-column runs, incompatible old model checkpoints are intentionally ignored.


In [ ]:
# 1A. User-editable configuration
from pathlib import Path

PIPELINE_VERSION = "anaemia_research_v3_1_40cols_2026_09_05"
FEATURE_BUILDER_VERSION = "domain_features_v8_structural_missingness"
RANDOM_STATE = 42
PILOT_TEST_EXPOSED = True
PAPER_SPLIT_RANDOM_STATE = 20260905
PROBABILITY_POLICY_VERSION = "clip_renormalize_float64_v1"
RUN_MODE = os.environ.get("ANAEMIA_RUN_MODE", "development")
STRICT_PAPER_MODE = True
REUSE_VALID_ARTIFACTS = True
FORCE_STAGES = set()               # e.g. {"splits", "lightgbm", "bootstrap"}
USE_GPU = False
# Keep False on the first paper pass. Set True only after nested and geographic
# validation manifests are complete and no modelling decision will change.
UNLOCK_FINAL_TEST = False

# Environment variables are convenient for local/CI smoke tests. In Colab,
# DATA_PATH=None safely discovers the final CSV from the locations below.
DATA_PATH = os.environ.get("ANAEMIA_DATA_PATH") or None
EXTERNAL_NFHS4_PATH = os.environ.get("ANAEMIA_NFHS4_PATH") or None

if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/anaemia_project_40cols_v31")
    DATA_CANDIDATES = [
        Path("/content/drive/MyDrive/anaemia_women_research_40cols.csv"),
        Path("/content/drive/MyDrive/NFHS_Project/anaemia_women_research_40cols.csv"),
        Path("/content/drive/MyDrive/NFHS_Project/data/anaemia_women_research_40cols.csv"),
        Path("/content/drive/MyDrive/nfhs_final.parquet"),
        Path("/content/drive/MyDrive/NFHS_Project/nfhs_final.parquet"),
        Path("/content/drive/MyDrive/IAIR7EFL.SAV"),
        Path("/content/drive/MyDrive/NFHS_Project/IAIR7EFL.SAV"),
    ]
else:
    PROJECT_DIR = Path.cwd() / "anaemia_project_40cols_v31"
    DATA_CANDIDATES = [
        Path.cwd() / "anaemia_women_research_40cols.csv",
        Path.cwd() / "nfhs_final.parquet",
        Path.cwd() / "IAIR7EFL.SAV",
    ]

MODE_SETTINGS = {
    "development": {
        "hpo_trials": 5,
        "hpo_rows": 80_000,
        "cv_splits": 3,
        "bootstrap_runs": 50,
        "shap_rows": 750,
        "nested_cv": False,
        "nested_outer": 3,
        "nested_inner": 3,
        "nested_trials": 3,
        "geographic_cv": False,
    },
    "paper": {
        "hpo_trials": 20,
        "hpo_rows": 200_000,
        "cv_splits": 5,
        "bootstrap_runs": 500,
        "shap_rows": 3_000,
        "nested_cv": True,
        "nested_outer": 5,
        "nested_inner": 3,
        "nested_trials": 8,
        "geographic_cv": True,
    },
}
if RUN_MODE not in MODE_SETTINGS:
    raise ValueError(f"RUN_MODE must be one of {list(MODE_SETTINGS)}")
CFG = MODE_SETTINGS[RUN_MODE]
SPLIT_RANDOM_STATE = (
    PAPER_SPLIT_RANDOM_STATE if RUN_MODE == "paper" else RANDOM_STATE
)

ENABLED_MODELS = [
    "dummy",
    "ordinal_logit",
    "multinomial_logit",
    "random_forest",
    "lightgbm",
    "xgboost",
    "catboost",
]
HPO_MODELS = ["random_forest", "lightgbm", "xgboost", "catboost"]
PRIMARY_FEATURE_SET = "policy"

TEST_FRACTION = 0.20
CALIBRATION_FRACTION = 0.10
HASH_MODE = "full" if RUN_MODE == "paper" else "fast"
MIN_SUBGROUP_N = 500
MIN_SUBGROUP_SEVERE = 20

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_DIR)
print("Run mode:", RUN_MODE, CFG)
print("Final-test unlock requested:", UNLOCK_FINAL_TEST)
print("Split random state:", SPLIT_RANDOM_STATE)


In [ ]:
# 1B. Imports, deterministic settings and plotting style
import json
import math
import time
import random
import hashlib
import logging
import platform
import warnings
import datetime as dt
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize_scalar
from scipy.stats import friedmanchisquare, wilcoxon, spearmanr

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.auto import tqdm

import sklearn
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, log_loss, cohen_kappa_score, mean_absolute_error,
    roc_curve, precision_recall_curve,
)
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm
import xgboost
import catboost
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler
import shap

warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
N_JOBS = max(1, (os.cpu_count() or 2) - 1)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("anaemia_v2")

CLASS_LABELS = {
    0: "No anaemia",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
}
CLASS_ORDER = [0, 1, 2, 3]
CLASS_COLORS = {
    0: "#2E8B57",
    1: "#E9C46A",
    2: "#F4A261",
    3: "#D62828",
}


SUPPORTED_PYTHON_MINORS = {(3, 12), (3, 13)}
RUNTIME_VERSIONS = {
    "python": platform.python_version(),
    "python_supported_by_notebook": sys.version_info[:2] in SUPPORTED_PYTHON_MINORS,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "lightgbm": lightgbm.__version__,
    "xgboost": xgboost.__version__,
    "catboost": catboost.__version__,
    "optuna": optuna.__version__,
    "shap": shap.__version__,
}
if not RUNTIME_VERSIONS["python_supported_by_notebook"]:
    warnings.warn(
        f"Python {platform.python_version()} is outside the tested 3.12/3.13 range."
    )
print("Runtime versions:", json.dumps(RUNTIME_VERSIONS, sort_keys=True))


## 2. Exact final-CSV data contract

The publication run requires the exact 40 raw fields below. Raw survey codes
remain unchanged in the CSV; recoding and feature construction occur here so
the complete analytical lineage is auditable.

| Role | Raw variables |
|---|---|
| Survey IDs/design/geography | `v002`, `v003`, `v005`, `v021`, `v022`, `v024`, `sdist` |
| Socio-demographic | `v012`, `v025`, `v501`, `v130`, `s116`, `v190`, `v133` |
| Fertility/reproductive | `v201`, `v208`, `v212`, `v213`, `v228`, `v312`, `v404`, `v405` |
| Anthropometry/access/environment | `v445`, `v467b`, `v467c`, `v467d`, `v467f`, `v113`, `v116`, `v161`, `v481` |
| Nutrition/comorbidity | `s728a`, `s731b`, `s731c`, `s731d`, `s731e`, `s731f`, `s731g` |
| Sensitivity outcome / target | `v456`, `v457` |

`v001` is excluded because it was identical to `v021` for all 724,115 rows.
`v106` is replaced by the more granular `v133` education-years field, and
`v136` was removed to prioritise stronger nutritional/reproductive domains.
`v228` means ever had a terminated pregnancy; current amenorrhea is `v405`.

Identifiers, weights, strata, district, haemoglobin and the target are never
allowed into the predictor matrix unless explicitly designated (state is used
only in the India policy feature set).


In [ ]:
# 2A. Exact final 40-column data contract
TARGET_SOURCE = "v457"
FINAL_RAW_COLUMNS = [
    "v002", "v003", "v005", "v021", "v022", "v024", "sdist",
    "v012", "v025", "v501", "v130", "s116", "v190", "v133",
    "v201", "v208", "v212", "v213", "v228", "v312", "v404", "v405",
    "v445", "v467b", "v467c", "v467d", "v467f",
    "v113", "v116", "v161", "v481",
    "s728a", "s731b", "s731c", "s731d", "s731e", "s731f", "s731g",
    "v456", "v457",
]
assert len(FINAL_RAW_COLUMNS) == 40
assert len(set(FINAL_RAW_COLUMNS)) == 40

REQUESTED_RAW_COLUMNS = list(FINAL_RAW_COLUMNS)
PAPER_REQUIRED = set(FINAL_RAW_COLUMNS)
PAPER_REQUIRED_ONE_OF = []
STRONGLY_RECOMMENDED = set()

EXPECTED_SOURCE_ROWS = 724_115
EXPECTED_STATE_COUNT = 36
EXPECTED_STATE_DISTRICT_PAIRS = 707
EXPECTED_V457_COUNTS = {1: 18_221, 2: 195_685, 3: 207_855, 4: 302_354}

RAW_ALLOWED_CODES = {
    "v457": {1, 2, 3, 4},
    "s116": {1, 2, 3, 4, 8},
    "v404": {0, 1}, "v405": {0, 1}, "v481": {0, 1}, "v228": {0, 1},
    "s728a": {0, 1, 8},
    "s731b": {0, 1, 2, 3}, "s731c": {0, 1, 2, 3},
    "s731d": {0, 1, 2, 3}, "s731e": {0, 1, 2, 3},
    "s731f": {0, 1, 2, 3}, "s731g": {0, 1, 2, 3},
    "v467b": {0, 1, 2}, "v467c": {0, 1, 2},
    "v467d": {0, 1, 2}, "v467f": {0, 1, 2},
}

OUTCOME_DERIVED_COLUMNS = {
    "v455", "v456", "v457", "v457a",
    "haemoglobin", "hemoglobin", "hb", "hb_level",
    "adjusted_hb_gdl", "anaemia_level", "anemia_level", "who2024_level",
}

FEATURE_DICTIONARY = {
    "v012": "Age in completed years",
    "v025": "Place of residence",
    "v024": "State/UT code",
    "sdist": "District code (validation metadata; not a primary predictor)",
    "v501": "Current marital status",
    "v130": "Religion",
    "s116": "Scheduled caste/tribe/OBC/none category; code 8 is unknown",
    "v190": "Household wealth quintile",
    "v133": "Education in completed single years",
    "v201": "Children ever born",
    "v208": "Births in preceding five years",
    "v212": "Age at first birth",
    "v213": "Currently pregnant",
    "v228": "Ever had a terminated pregnancy",
    "v312": "Current contraceptive method",
    "v404": "Currently breastfeeding",
    "v405": "Currently amenorrheic",
    "v445": "Body mass index; raw DHS storage is 100×BMI",
    "v467b": "Permission to seek treatment barrier",
    "v467c": "Money needed for treatment barrier",
    "v467d": "Distance to health facility barrier",
    "v467f": "Not wanting to go alone barrier",
    "v113": "Source of drinking water",
    "v116": "Type of toilet facility",
    "v161": "Household cooking fuel",
    "v481": "Covered by health insurance",
    "s728a": "Self-reported current diabetes; code 8 is don't know",
    "s731b": "Frequency of eating pulses or beans",
    "s731c": "Frequency of eating dark green leafy vegetables",
    "s731d": "Frequency of eating fruits",
    "s731e": "Frequency of eating eggs",
    "s731f": "Frequency of eating fish",
    "s731g": "Frequency of eating chicken or meat",
    "v456": "Adjusted haemoglobin (sensitivity analysis only)",
    "v457": "Raw four-level anaemia outcome",
}

for var in FINAL_RAW_COLUMNS:
    role = FEATURE_DICTIONARY.get(var, "Survey design / respondent identifier")
    print(f"{var:<8} {role}")


In [ ]:
# 2B. File discovery, hashing and raw-data loader
def inspect_source_columns(path):
    # Read only source metadata/header; never load the complete wide file.
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".sav":
        import pyreadstat
        _, meta = pyreadstat.read_sav(str(path), metadataonly=True)
        return [str(c).lower() for c in meta.column_names]
    if suffix == ".dta":
        reader = pd.read_stata(path, iterator=True, convert_categoricals=False)
        try:
            return [str(c).lower() for c in reader.varlist]
        finally:
            reader.close()
    if suffix == ".parquet":
        import pyarrow.parquet as pq
        return [str(c).lower() for c in pq.ParquetFile(path).schema_arrow.names]
    if suffix == ".csv":
        return [str(c).lower() for c in pd.read_csv(path, nrows=0).columns]
    return []


def locate_data_file(explicit_path=None, candidates=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f"Configured DATA_PATH does not exist: {path}")
        return path
    found = [Path(p) for p in (candidates or []) if Path(p).exists()]
    if not found:
        raise FileNotFoundError(
            "No compatible NFHS-5 file was found. Upload "
            "anaemia_women_research_40cols.csv to Drive or set DATA_PATH explicitly."
        )
    # Prefer sources that satisfy the scientific contract; use file type only
    # as a tie-breaker. This prevents an old narrow CSV from shadowing a valid
    # raw-like Parquet export in the same Drive.
    priority = {".sav": 0, ".dta": 1, ".parquet": 2, ".csv": 3}
    def candidate_rank(path):
        try:
            cols = set(inspect_source_columns(path))
        except Exception as exc:
            logger.warning("Could not inspect candidate %s: %s", path, exc)
            cols = set()
        required_complete = PAPER_REQUIRED.issubset(cols)
        exact_final_schema = cols == set(FINAL_RAW_COLUMNS)
        contract_hits = len(PAPER_REQUIRED & cols) + sum(
            bool(group & cols) for group in PAPER_REQUIRED_ONE_OF
        )
        recommended_hits = len(STRONGLY_RECOMMENDED & cols)
        return (
            not exact_final_schema,
            not required_complete,
            -contract_hits,
            -recommended_hits,
            priority.get(path.suffix.lower(), 9),
            "anaemia_women_final" in path.name.lower(),
        )
    found.sort(key=candidate_rank)
    return found[0]


def atomic_write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)
    os.replace(tmp, path)


def atomic_joblib_dump(obj, path, compress=3):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    joblib.dump(obj, tmp, compress=compress)
    os.replace(tmp, path)


def atomic_to_parquet(frame, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(tmp, index=index)
    os.replace(tmp, path)


def atomic_savez(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)


def stable_hash(payload):
    raw = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def sha256_file(path, mode="full", chunk_size=8 * 1024 * 1024):
    path = Path(path)
    stat = path.stat()
    cache_path = PROJECT_DIR / "file_hash_cache.json"
    cache = {}
    if cache_path.exists():
        try:
            cache = json.loads(cache_path.read_text())
        except Exception:
            cache = {}
    key = f"{path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}|{mode}"
    if key in cache:
        return cache[key]

    h = hashlib.sha256()
    h.update(str(stat.st_size).encode())
    if mode == "fast":
        with open(path, "rb") as f:
            h.update(f.read(chunk_size))
            if stat.st_size > chunk_size:
                f.seek(max(0, stat.st_size - chunk_size))
                h.update(f.read(chunk_size))
    else:
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                h.update(chunk)
    digest = h.hexdigest()
    cache[key] = digest
    atomic_write_json(cache, cache_path)
    return digest


def _read_sav_selected(path, requested_lower):
    import pyreadstat
    _, meta = pyreadstat.read_sav(str(path), metadataonly=True)
    original = list(meta.column_names)
    lower_to_original = {str(c).lower(): c for c in original}
    usecols = [lower_to_original[c] for c in requested_lower if c in lower_to_original]
    if "v457" not in lower_to_original:
        raise KeyError("v457 is not present in the supplied SAV file.")
    frame, meta = pyreadstat.read_sav(
        str(path), usecols=usecols, apply_value_formats=False
    )
    frame.columns = [str(c).lower() for c in frame.columns]
    labels = {}
    variable_value_labels = getattr(meta, "variable_value_labels", {}) or {}
    for original_name, mapping in variable_value_labels.items():
        labels[str(original_name).lower()] = {
            str(k): str(v) for k, v in (mapping or {}).items()
        }
    return frame, labels


def load_nfhs_source(path):
    path = Path(path)
    suffix = path.suffix.lower()
    labels = {}
    if suffix == ".sav":
        frame, labels = _read_sav_selected(path, set(REQUESTED_RAW_COLUMNS))
    elif suffix == ".dta":
        frame = pd.read_stata(path, convert_categoricals=False)
        frame.columns = frame.columns.str.lower()
        keep = [c for c in REQUESTED_RAW_COLUMNS if c in frame.columns]
        frame = frame[keep].copy()
    elif suffix == ".parquet":
        source_columns = inspect_source_columns(path)
        lower_to_original = {str(c).lower(): c for c in source_columns}
        keep = [
            lower_to_original[c] for c in REQUESTED_RAW_COLUMNS
            if c in lower_to_original
        ]
        frame = pd.read_parquet(path, columns=keep)
        frame.columns = frame.columns.str.lower()
    elif suffix == ".csv":
        source_columns = inspect_source_columns(path)
        requested = set(REQUESTED_RAW_COLUMNS)
        frame = pd.read_csv(
            path,
            usecols=lambda c: str(c).lower() in requested,
            low_memory=False,
        )
        frame.columns = frame.columns.str.lower()
    else:
        raise ValueError(f"Unsupported data format: {suffix}")
    return frame, labels


SOURCE_PATH = locate_data_file(DATA_PATH, DATA_CANDIDATES)
SOURCE_AVAILABLE_COLUMNS = set(inspect_source_columns(SOURCE_PATH))
SOURCE_HASH = sha256_file(SOURCE_PATH, HASH_MODE)
RUN_CONFIG_FOR_HASH = {
    "pipeline_version": PIPELINE_VERSION,
    "source_hash": SOURCE_HASH,
    # Development and publication runs must never append trials/folds to the
    # same studies. Trial *targets* are deliberately excluded so a larger
    # target resumes the same scientifically compatible study.
    "run_mode": RUN_MODE,
    "hpo_rows": CFG["hpo_rows"],
    "cv_splits": CFG["cv_splits"],
    "target_mapping": {1: 3, 2: 2, 3: 1, 4: 0},
    "final_raw_columns": FINAL_RAW_COLUMNS,
    "feature_builder_version": FEATURE_BUILDER_VERSION,
    "random_state": RANDOM_STATE,
    "split_random_state": SPLIT_RANDOM_STATE,
    "pilot_test_exposed": PILOT_TEST_EXPOSED,
    "probability_policy": PROBABILITY_POLICY_VERSION,
    "primary_feature_set": PRIMARY_FEATURE_SET,
    "test_fraction": TEST_FRACTION,
    "calibration_fraction": CALIBRATION_FRACTION,
}
RUN_FINGERPRINT = stable_hash(RUN_CONFIG_FOR_HASH)[:16]
RUN_DIR = PROJECT_DIR / "runs" / RUN_FINGERPRINT
DIRS = {
    "cache": RUN_DIR / "cache",
    "models": RUN_DIR / "models",
    "studies": RUN_DIR / "studies",
    "predictions": RUN_DIR / "predictions",
    "tables": RUN_DIR / "tables",
    "figures": RUN_DIR / "figures",
    "reports": RUN_DIR / "reports",
    "logs": RUN_DIR / "logs",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

OPTUNA_DB = f"sqlite:///{(DIRS['studies'] / 'optuna.db').as_posix()}"
print("Source:", SOURCE_PATH)
print("Source fingerprint:", SOURCE_HASH[:16])
print("Run fingerprint:", RUN_FINGERPRINT)
print("Run directory:", RUN_DIR)


In [ ]:
# 2C. Stage manifests: reuse only compatible artifacts
def stage_manifest_path(stage):
    return DIRS["cache"] / f"{stage}.manifest.json"


def stage_signature(stage, parameters=None):
    return stable_hash({
        "stage": stage,
        "pipeline_version": PIPELINE_VERSION,
        "run_fingerprint": RUN_FINGERPRINT,
        "parameters": parameters or {},
    })


def stage_is_valid(stage, outputs, parameters=None):
    if not REUSE_VALID_ARTIFACTS or stage in FORCE_STAGES:
        return False
    manifest_path = stage_manifest_path(stage)
    outputs = [Path(p) for p in outputs]
    if not manifest_path.exists() or not all(p.exists() for p in outputs):
        return False
    try:
        meta = json.loads(manifest_path.read_text())
    except Exception:
        return False
    return (
        meta.get("signature") == stage_signature(stage, parameters)
        and meta.get("status") == "complete"
    )


def mark_stage_complete(stage, outputs, parameters=None, extra=None):
    outputs = [str(Path(p)) for p in outputs]
    meta = {
        "stage": stage,
        "status": "complete",
        "signature": stage_signature(stage, parameters),
        "run_fingerprint": RUN_FINGERPRINT,
        "pipeline_version": PIPELINE_VERSION,
        "completed_at": dt.datetime.now(dt.timezone.utc).isoformat(),
        "outputs": outputs,
        "extra": extra or {},
    }
    atomic_write_json(meta, stage_manifest_path(stage))


RUN_MANIFEST = {
    "pipeline_version": PIPELINE_VERSION,
    "run_fingerprint": RUN_FINGERPRINT,
    "source_path": str(SOURCE_PATH),
    "source_hash": SOURCE_HASH,
    "hash_mode": HASH_MODE,
    "run_mode": RUN_MODE,
    "configuration": RUN_CONFIG_FOR_HASH,
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
        **RUNTIME_VERSIONS,
    },
    "created_at": dt.datetime.now(dt.timezone.utc).isoformat(),
}
atomic_write_json(RUN_MANIFEST, RUN_DIR / "run_manifest.json")
PROTOCOL_AMENDMENT = {
    "version": "v3.1-post-pilot-test-exposure",
    "pilot_test_exposed": PILOT_TEST_EXPOSED,
    "paper_split_random_state": PAPER_SPLIT_RANDOM_STATE,
    "reason": (
        "The v3 development run evaluated its test partition. v3.1 treats "
        "those results as pilot-only, changes the paper split seed before "
        "paper modelling, and requires external validation for publication."
    ),
    "decision_freeze": (
        "After final-test unlock, do not change features, models, endpoints, "
        "thresholds, calibration, or subgroup rules."
    ),
}
atomic_write_json(
    PROTOCOL_AMENDMENT, DIRS["reports"] / "protocol_amendment_v31.json"
)

legacy_dir = PROJECT_DIR.parent / "anaemia_project" / "models" / "checkpoints"
if legacy_dir.exists():
    print(
        "Legacy checkpoint directory detected but intentionally ignored because "
        "its artifacts do not carry the corrected pipeline fingerprint:", legacy_dir
    )
print("Checkpoint manager ready.")


In [ ]:
# 2D. Load selected fields and enforce the exact final-CSV contract
raw_cache = DIRS["cache"] / "raw_selected_40cols.parquet"
labels_cache = DIRS["cache"] / "source_value_labels.json"
profile_cache = DIRS["tables"] / "raw_column_profile.parquet"
raw_stage_params = {
    "requested_columns": FINAL_RAW_COLUMNS,
    "expected_rows": EXPECTED_SOURCE_ROWS,
    "schema_version": "final_40cols_v1",
}

if stage_is_valid(
    "raw_selected", [raw_cache, labels_cache, profile_cache], raw_stage_params
):
    raw_df = pd.read_parquet(raw_cache)
    SOURCE_VALUE_LABELS = json.loads(labels_cache.read_text())
    raw_profile = pd.read_parquet(profile_cache)
    print("Loaded compatible selected-data checkpoint:", raw_cache)
else:
    raw_df, SOURCE_VALUE_LABELS = load_nfhs_source(SOURCE_PATH)
    raw_df.columns = raw_df.columns.str.lower()
    missing_loaded = sorted(set(FINAL_RAW_COLUMNS) - set(raw_df.columns))
    if missing_loaded:
        raise KeyError(f"Final 40-column source is missing: {missing_loaded}")
    raw_df = raw_df[FINAL_RAW_COLUMNS].copy()
    raw_df.insert(0, "row_id", np.arange(len(raw_df), dtype=np.int64))
    raw_profile = pd.DataFrame([
        {
            "variable": c,
            "role": FEATURE_DICTIONARY.get(c, "survey_design_or_identifier"),
            "dtype": str(raw_df[c].dtype),
            "missing_n": int(raw_df[c].isna().sum()),
            "missing_percent": float(100 * raw_df[c].isna().mean()),
            "unique_n": int(raw_df[c].nunique(dropna=True)),
            "minimum": pd.to_numeric(raw_df[c], errors="coerce").min(),
            "maximum": pd.to_numeric(raw_df[c], errors="coerce").max(),
        }
        for c in FINAL_RAW_COLUMNS
    ])
    atomic_to_parquet(raw_df, raw_cache, index=False)
    atomic_to_parquet(raw_profile, profile_cache, index=False)
    atomic_write_json(SOURCE_VALUE_LABELS, labels_cache)
    mark_stage_complete(
        "raw_selected", [raw_cache, labels_cache, profile_cache], raw_stage_params,
        extra={"rows": len(raw_df), "columns": FINAL_RAW_COLUMNS},
    )
    print("Created selected-data checkpoint:", raw_cache)

available = set(raw_df.columns)
source_column_order = inspect_source_columns(SOURCE_PATH)
source_available = set(source_column_order)
missing_required = sorted(set(FINAL_RAW_COLUMNS) - source_available)
unexpected_source_columns = sorted(source_available - set(FINAL_RAW_COLUMNS))

quality_flags = []
if missing_required:
    quality_flags.append({
        "severity": "blocking_for_publication",
        "issue": "final_schema_columns_missing",
        "columns": missing_required,
        "action": "Use anaemia_women_research_40cols.csv or re-extract these fields.",
    })

exact_schema_order = source_column_order == FINAL_RAW_COLUMNS
if SOURCE_PATH.suffix.lower() == ".csv" and not exact_schema_order:
    quality_flags.append({
        "severity": "blocking_for_publication",
        "issue": "final_csv_schema_or_order_mismatch",
        "action": "Use the validated final 40-column CSV without manual column changes.",
    })

raw_blank_cells = int(raw_df[FINAL_RAW_COLUMNS].isna().sum().sum())
if raw_blank_cells:
    quality_flags.append({
        "severity": "blocking_for_publication",
        "issue": "unexpected_blank_raw_cells",
        "count": raw_blank_cells,
        "action": "Revalidate the final CSV; raw special codes must not be silently blanked.",
    })

invalid_code_counts = {}
for col, valid_codes in RAW_ALLOWED_CODES.items():
    values = pd.to_numeric(raw_df[col], errors="coerce")
    invalid = values.notna() & ~values.isin(valid_codes)
    invalid_code_counts[col] = int(invalid.sum())
    if invalid.any():
        quality_flags.append({
            "severity": "blocking_for_publication",
            "issue": f"unexpected_codes_{col}",
            "count": int(invalid.sum()),
            "values": sorted(values.loc[invalid].unique().tolist()),
            "action": "Verify the NFHS-5 recode documentation before modelling.",
        })

target_raw_counts = {
    int(k): int(v)
    for k, v in pd.to_numeric(raw_df["v457"], errors="coerce")
    .value_counts().sort_index().items()
}
profile_checks = {
    "row_count": len(raw_df) == EXPECTED_SOURCE_ROWS,
    "target_counts": target_raw_counts == EXPECTED_V457_COUNTS,
    "state_count": raw_df["v024"].nunique() == EXPECTED_STATE_COUNT,
    "state_district_pairs": (
        raw_df[["v024", "sdist"]].drop_duplicates().shape[0]
        == EXPECTED_STATE_DISTRICT_PAIRS
    ),
}
for check_name, passed in profile_checks.items():
    if not passed:
        quality_flags.append({
            "severity": "blocking_for_publication",
            "issue": f"known_final_csv_profile_failed_{check_name}",
            "action": "Confirm that the validated final CSV is being used.",
        })

id_columns = ["v024", "v021", "v002", "v003"]
id_frame = raw_df[id_columns].apply(pd.to_numeric, errors="coerce")
valid_id = id_frame.notna().all(axis=1)
duplicate_ids = int(id_frame.loc[valid_id].duplicated(keep=False).sum())
if duplicate_ids:
    quality_flags.append({
        "severity": "blocking_for_publication",
        "issue": "duplicate_composite_respondent_ids",
        "rows_affected": duplicate_ids,
        "action": "Inspect identifiers; do not deduplicate automatically.",
    })

unknown_code_counts = {
    "caste_s116_code_8": int(pd.to_numeric(raw_df["s116"], errors="coerce").eq(8).sum()),
    "diabetes_s728a_code_8": int(pd.to_numeric(raw_df["s728a"], errors="coerce").eq(8).sum()),
}

contract_report = {
    "source_path": str(SOURCE_PATH),
    "source_kind": "validated_final_40_column_extract",
    "expected_columns": FINAL_RAW_COLUMNS,
    "source_column_order": source_column_order,
    "exact_schema_order": exact_schema_order,
    "unexpected_source_columns": unexpected_source_columns,
    "missing_required": missing_required,
    "paper_contract_passed": not any(
        flag["severity"] == "blocking_for_publication" for flag in quality_flags
    ),
    "survey_design_available": {"v005", "v021", "v022"}.issubset(available),
    "raw_rows": len(raw_df),
    "raw_blank_cells": raw_blank_cells,
    "target_raw_counts": target_raw_counts,
    "invalid_code_counts": invalid_code_counts,
    "profile_checks": profile_checks,
    "state_count": int(raw_df["v024"].nunique()),
    "state_district_pairs": int(raw_df[["v024", "sdist"]].drop_duplicates().shape[0]),
    "identifier_audit": {
        "id_definition": id_columns,
        "complete_id_rows": int(valid_id.sum()),
        "rows_in_duplicated_id_groups": duplicate_ids,
    },
    "unknown_code_counts_recode_to_missing": unknown_code_counts,
    "quality_flags": quality_flags,
}
atomic_write_json(contract_report, DIRS["reports"] / "data_contract.json")

SURVEY_DESIGN_AVAILABLE = contract_report["survey_design_available"]
DESCRIPTIVE_LABEL = "Survey-weighted" if SURVEY_DESIGN_AVAILABLE else "Unweighted"

print("Raw selected shape:", raw_df.shape)
print("Exact final schema/order:", exact_schema_order)
print("State–district domains:", contract_report["state_district_pairs"])
print("Duplicate composite respondent IDs:", duplicate_ids)
print("Target counts:", target_raw_counts)
for flag in quality_flags:
    print(" -", flag["severity"], "|", flag["issue"])

if STRICT_PAPER_MODE and RUN_MODE == "paper" and not contract_report["paper_contract_passed"]:
    raise RuntimeError(
        "Publication data contract failed. Use the validated "
        "anaemia_women_research_40cols.csv unchanged. See data_contract.json."
    )


## 3. Correct outcome and domain-safe feature engineering

The NFHS target is mapped explicitly: `v457` raw codes `4,3,2,1` become
`0 No anaemia, 1 Mild, 2 Moderate, 3 Severe`. `v456` adjusted haemoglobin is
reserved exclusively for threshold sensitivity and is prohibited from every
predictor set.

The builder recodes special values to missing, converts BMI and survey-weight
storage scales, converts food frequency codes to an ordered frequency score,
and creates a small pre-specified set of fertility–nutrition interactions.
No feature is selected using the locked test set or SHAP results.


In [ ]:
# 3A. Cleaning helpers and deterministic final-CSV feature builder
TARGET_MAP_DHS_1_TO_4 = {1: 3, 2: 2, 3: 1, 4: 0}
FOOD_FREQUENCY_SCORE = {0: 0.0, 3: 1.0, 2: 2.0, 1: 3.0}


def numeric(series):
    if series is None:
        return pd.Series(dtype=float)
    return pd.to_numeric(series, errors="coerce")


def valid_numeric(series, lower=None, upper=None, invalid_values=(), index=None):
    if series is None:
        return pd.Series(np.nan, index=index, dtype=float)
    s = numeric(series).replace(list(invalid_values), np.nan)
    if lower is not None:
        s = s.where(s >= lower)
    if upper is not None:
        s = s.where(s <= upper)
    return s.astype(float)


def valid_category(
    series, valid_values=None, invalid_values=(8, 9, 98, 99, 998, 999), index=None
):
    if series is None:
        return pd.Series(np.nan, index=index, dtype=object)
    s = numeric(series).replace(list(invalid_values), np.nan)
    if valid_values is not None:
        s = s.where(s.isin(valid_values))
    return s.astype("object")


def barrier_indicator(series, index=None):
    if series is None:
        return pd.Series(np.nan, index=index, dtype=float)
    # 0=no problem; 1=big problem; 2=not a big problem.
    return numeric(series).map({0: 0.0, 1: 1.0, 2: 0.0})


def food_frequency_score(series, index=None):
    if series is None:
        return pd.Series(np.nan, index=index, dtype=float)
    # Raw: 0 never, 1 daily, 2 weekly, 3 occasionally.
    return numeric(series).map(FOOD_FREQUENCY_SCORE).astype(float)


def decode_bmi(series, index=None):
    if series is None:
        return pd.Series(np.nan, index=index, dtype=float)
    s = numeric(series)
    if pd.notna(s.dropna().median()) and s.dropna().median() > 100:
        s = s / 100.0
    return s.where(s.between(10, 60)).astype(float)


def survey_weight(series, index=None):
    if series is None:
        return pd.Series(1.0, index=index, dtype=float)
    s = numeric(series)
    if pd.notna(s.dropna().median()) and s.dropna().median() > 100:
        s = s / 1_000_000.0
    return s.where(s > 0).astype(float)


def derive_region_group(state_code):
    code = numeric(state_code)
    out = pd.Series("Other India", index=code.index, dtype="object")
    out.loc[code.isin([8, 9, 10, 23])] = "BIMARU"
    # Andhra Pradesh, Karnataka, Kerala, Tamil Nadu and Telangana.
    out.loc[code.isin([28, 29, 32, 33, 36])] = "Southern states"
    out.loc[code.isna()] = np.nan
    return out


def construct_analysis_frame(raw):
    raw = raw.copy()
    idx = raw.index
    out = pd.DataFrame(index=idx)
    out["row_id"] = raw.get("row_id", pd.Series(np.arange(len(raw)), index=idx))

    v457 = numeric(raw.get("v457"))
    out["anaemia_level"] = v457.map(TARGET_MAP_DHS_1_TO_4)
    out["raw_v457"] = v457

    # Survey design and audit metadata: never predictors except state_code in
    # the explicitly labelled India policy model.
    out["sample_weight"] = survey_weight(raw.get("v005"), index=idx)
    out["state_code"] = valid_numeric(raw.get("v024"), 1, 99, index=idx)
    out["district_code"] = valid_numeric(raw.get("sdist"), 1, 999, index=idx)
    psu_raw = numeric(raw.get("v021")).reindex(idx)
    strata_raw = numeric(raw.get("v022")).reindex(idx)
    state_token = out["state_code"].fillna(-1).astype("Int64").astype(str)
    out["psu"] = state_token + "|" + psu_raw.astype("Int64").astype(str)
    out.loc[psu_raw.isna(), "psu"] = "row|" + out.loc[psu_raw.isna(), "row_id"].astype(str)
    out["strata"] = state_token + "|" + strata_raw.astype("Int64").astype(str)
    out.loc[strata_raw.isna(), "strata"] = np.nan
    out["respondent_key"] = (
        state_token + "|" + psu_raw.astype("Int64").astype(str) + "|"
        + numeric(raw.get("v002")).astype("Int64").astype(str) + "|"
        + numeric(raw.get("v003")).astype("Int64").astype(str)
    )

    # Socio-demographic predictors.
    out["age"] = valid_numeric(raw.get("v012"), 15, 49, index=idx)
    out["residence"] = valid_category(raw.get("v025"), {1, 2}, index=idx)
    out["marital_status"] = valid_category(raw.get("v501"), {0, 1, 2, 3, 4, 5}, index=idx)
    out["religion"] = valid_category(raw.get("v130"), invalid_values=(98, 99, 998, 999), index=idx)
    out["caste"] = valid_category(raw.get("s116"), {1, 2, 3, 4}, index=idx)
    out["wealth_quintile"] = valid_category(raw.get("v190"), {1, 2, 3, 4, 5}, index=idx)
    out["education_years"] = valid_numeric(raw.get("v133"), 0, 20, index=idx)

    # Fertility and reproductive predictors.
    out["children_ever_born"] = valid_numeric(raw.get("v201"), 0, 25, index=idx)
    out["births_last_5y"] = valid_numeric(raw.get("v208"), 0, 10, index=idx)
    out["age_first_birth"] = valid_numeric(
        raw.get("v212"), 10, 49, invalid_values=(0,), index=idx
    )
    out["ever_given_birth"] = np.where(
        out["children_ever_born"].notna(),
        (out["children_ever_born"] > 0).astype(float),
        np.nan,
    )
    out["currently_pregnant"] = valid_category(raw.get("v213"), {0, 1}, index=idx)
    out["pregnancy_termination_history"] = valid_category(raw.get("v228"), {0, 1}, index=idx)
    out["contraceptive_method"] = valid_category(
        raw.get("v312"), set(range(0, 19)), invalid_values=(98, 99), index=idx
    )
    out["currently_breastfeeding"] = valid_category(raw.get("v404"), {0, 1}, index=idx)
    out["currently_amenorrheic"] = valid_category(raw.get("v405"), {0, 1}, index=idx)

    # Anthropometry, healthcare access and environment.
    out["bmi"] = decode_bmi(raw.get("v445"), index=idx)
    for source, name in [
        ("v467b", "barrier_permission"),
        ("v467c", "barrier_money"),
        ("v467d", "barrier_distance"),
        ("v467f", "barrier_go_alone"),
    ]:
        out[name] = barrier_indicator(raw.get(source), index=idx)
    barrier_cols = [
        "barrier_permission", "barrier_money", "barrier_distance", "barrier_go_alone"
    ]
    out["healthcare_barrier_count"] = out[barrier_cols].sum(axis=1, min_count=1)
    out["drinking_water"] = valid_category(
        raw.get("v113"), invalid_values=(98, 99, 998, 999), index=idx
    )
    out["toilet_facility"] = valid_category(
        raw.get("v116"), invalid_values=(98, 99, 998, 999), index=idx
    )
    out["cooking_fuel"] = valid_category(
        raw.get("v161"), invalid_values=(98, 99, 998, 999), index=idx
    )
    out["health_insurance"] = valid_category(raw.get("v481"), {0, 1}, index=idx)
    out["self_reported_diabetes"] = valid_category(raw.get("s728a"), {0, 1}, index=idx)

    # Food frequency: 0 never, 1 occasionally, 2 weekly, 3 daily after recoding.
    food_sources = {
        "s731b": "pulses_frequency", "s731c": "leafy_vegetable_frequency",
        "s731d": "fruit_frequency", "s731e": "egg_frequency",
        "s731f": "fish_frequency", "s731g": "chicken_meat_frequency",
    }
    for source, name in food_sources.items():
        out[name] = food_frequency_score(raw.get(source), index=idx)
    food_cols = list(food_sources.values())
    food_observed = out[food_cols].notna().sum(axis=1)
    out["diet_diversity_score"] = (
        (out[food_cols] > 0).sum(axis=1).astype(float).where(food_observed > 0)
    )
    out["diet_weekly_or_daily_count"] = (
        (out[food_cols] >= 2).sum(axis=1).astype(float).where(food_observed > 0)
    )
    animal_cols = ["egg_frequency", "fish_frequency", "chicken_meat_frequency"]
    out["animal_source_food_score"] = out[animal_cols].sum(axis=1, min_count=1)

    # Outcome sensitivity only. v214 is deliberately absent from the final
    # extract, therefore WHO sensitivity is complete for non-pregnant women and
    # excludes pregnant women whose trimester cannot be established.
    hb = numeric(raw.get("v456")).reindex(idx)
    if pd.notna(hb.dropna().median()) and hb.dropna().median() > 30:
        hb = hb / 10.0
    out["adjusted_hb_gdl"] = hb.where(hb.between(2.5, 20.0))
    out["pregnancy_month"] = np.nan

    # Pre-specified groups and fertility–nutrition interaction features.
    out["region_group"] = derive_region_group(out["state_code"])
    out["age_group"] = pd.cut(
        out["age"], [14, 19, 29, 39, 49], labels=["15–19", "20–29", "30–39", "40–49"]
    ).astype("object")
    out["education_group"] = pd.cut(
        out["education_years"], [-1, 0, 5, 10, 12, 20],
        labels=["None", "1–5 years", "6–10 years", "11–12 years", "13+ years"],
    ).astype("object")
    out["bmi_group"] = pd.cut(
        out["bmi"], [0, 18.5, 25, 30, np.inf], right=False,
        labels=["Underweight", "Normal", "Overweight", "Obesity"],
    ).astype("object")
    out["diet_diversity_group"] = pd.cut(
        out["diet_diversity_score"], [-1, 2, 4, 6],
        labels=["Low (0–2)", "Medium (3–4)", "High (5–6)"],
    ).astype("object")
    out["underweight"] = np.where(
        out["bmi"].notna(), (out["bmi"] < 18.5).astype(float), np.nan
    )
    out["high_parity"] = np.where(
        out["children_ever_born"].notna(),
        (out["children_ever_born"] >= 3).astype(float), np.nan,
    )
    out["early_first_birth"] = np.where(
        out["age_first_birth"].notna(), (out["age_first_birth"] < 18).astype(float), np.nan
    )
    out["low_diet_diversity"] = np.where(
        out["diet_diversity_score"].notna(),
        (out["diet_diversity_score"] <= 2).astype(float), np.nan,
    )
    parity_clip = out["children_ever_born"].clip(upper=8)
    out["parity_x_underweight"] = parity_clip * out["underweight"]
    out["parity_x_current_pregnancy"] = parity_clip * numeric(out["currently_pregnant"])
    out["births5_x_underweight"] = out["births_last_5y"] * out["underweight"]
    out["early_birth_x_parity"] = out["early_first_birth"] * parity_clip
    out["births5_x_breastfeeding"] = (
        out["births_last_5y"] * numeric(out["currently_breastfeeding"])
    )
    out["termination_x_high_parity"] = (
        numeric(out["pregnancy_termination_history"]) * out["high_parity"]
    )
    out["underweight_x_low_diet_diversity"] = (
        out["underweight"] * out["low_diet_diversity"]
    )
    return out


def assert_no_target_leakage(feature_columns):
    lower = {str(c).lower() for c in feature_columns}
    leaked = sorted(lower & OUTCOME_DERIVED_COLUMNS)
    if leaked:
        raise AssertionError(f"Target leakage detected in predictors: {leaked}")


In [ ]:
# 3B. Build/cache the corrected analysis frame
analysis_cache = DIRS["cache"] / "analysis_frame_40cols.parquet"
engineered_profile_path = DIRS["tables"] / "engineered_feature_profile.parquet"
analysis_params = {
    "target_map": TARGET_MAP_DHS_1_TO_4,
    "feature_dictionary": FEATURE_DICTIONARY,
    "builder_version": FEATURE_BUILDER_VERSION,
}

if stage_is_valid(
    "analysis_frame", [analysis_cache, engineered_profile_path], analysis_params
):
    analysis_df = pd.read_parquet(analysis_cache)
    engineered_profile = pd.read_parquet(engineered_profile_path)
    print("Loaded compatible analysis-frame checkpoint.")
else:
    analysis_df = construct_analysis_frame(raw_df)
    analysis_df = analysis_df.loc[analysis_df["anaemia_level"].notna()].copy()
    analysis_df["anaemia_level"] = analysis_df["anaemia_level"].astype("int8")
    analysis_df["row_id"] = analysis_df["row_id"].astype("int64")
    engineered_profile = pd.DataFrame([
        {
            "feature": c,
            "dtype": str(analysis_df[c].dtype),
            "missing_n": int(analysis_df[c].isna().sum()),
            "missing_percent": float(100 * analysis_df[c].isna().mean()),
            "unique_n": int(analysis_df[c].nunique(dropna=True)),
        }
        for c in analysis_df.columns
    ])
    atomic_to_parquet(analysis_df, analysis_cache, index=False)
    atomic_to_parquet(engineered_profile, engineered_profile_path, index=False)
    mark_stage_complete(
        "analysis_frame", [analysis_cache, engineered_profile_path], analysis_params,
        extra={"rows": len(analysis_df), "columns": list(analysis_df.columns)},
    )

observed_raw_codes = set(numeric(raw_df["v457"]).dropna().astype(int).unique())
if observed_raw_codes != {1, 2, 3, 4}:
    raise AssertionError(f"Unexpected v457 coding: {sorted(observed_raw_codes)}")
expected_mapping_check = pd.Series([1, 2, 3, 4]).map(TARGET_MAP_DHS_1_TO_4).tolist()
assert expected_mapping_check == [3, 2, 1, 0]
assert analysis_df["anaemia_level"].isin(CLASS_ORDER).all()
assert analysis_df["row_id"].is_unique
assert analysis_df["respondent_key"].is_unique
if STRICT_PAPER_MODE and RUN_MODE == "paper":
    assert len(analysis_df) == EXPECTED_SOURCE_ROWS

counts = analysis_df["anaemia_level"].value_counts().sort_index()
target_audit = pd.DataFrame({
    "class_code": CLASS_ORDER,
    "label": [CLASS_LABELS[c] for c in CLASS_ORDER],
    "count": [int(counts.get(c, 0)) for c in CLASS_ORDER],
})
target_audit["unweighted_percent"] = 100 * target_audit["count"] / len(analysis_df)
display(target_audit)
atomic_to_parquet(target_audit, DIRS["tables"] / "target_audit.parquet")
print("Corrected analysis shape:", analysis_df.shape)


## 4. Survey-weighted descriptive analysis

Unweighted counts describe the analytic sample. Survey-weighted percentages
estimate the represented population. EDA covers the pre-specified demographic,
fertility, nutrition, insurance and reproductive-health domains; it does not
screen variables for later inclusion based on significance.


In [ ]:
# 4A. Weighted prevalence and target distribution figure
def normalized_weights(weights):
    w = np.asarray(weights, dtype=float)
    valid = np.isfinite(w) & (w > 0)
    if not valid.all():
        replacement = np.nanmedian(w[valid]) if valid.any() else 1.0
        w = np.where(valid, w, replacement)
    return w / np.mean(w)


def weighted_class_prevalence(y, weights):
    y = np.asarray(y)
    w = normalized_weights(weights)
    return {
        c: float(np.sum(w[y == c]) / np.sum(w))
        for c in CLASS_ORDER
    }


weighted_prev = weighted_class_prevalence(
    analysis_df["anaemia_level"],
    analysis_df["sample_weight"],
)
unweighted_prev = analysis_df["anaemia_level"].value_counts(normalize=True).to_dict()

prevalence_table = pd.DataFrame({
    "class": [CLASS_LABELS[c] for c in CLASS_ORDER],
    "unweighted_n": [int(counts.get(c, 0)) for c in CLASS_ORDER],
    "unweighted_percent": [100 * unweighted_prev.get(c, 0) for c in CLASS_ORDER],
    "survey_weighted_percent": [100 * weighted_prev.get(c, 0) for c in CLASS_ORDER],
})
display(prevalence_table.round(3))
atomic_to_parquet(prevalence_table, DIRS["tables"] / "weighted_prevalence.parquet")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].bar(
    prevalence_table["class"], prevalence_table["unweighted_percent"],
    color=[CLASS_COLORS[c] for c in CLASS_ORDER]
)
axes[0].set_title("Analytic sample distribution")
axes[0].set_ylabel("Percent")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(
    prevalence_table["class"], prevalence_table["survey_weighted_percent"],
    color=[CLASS_COLORS[c] for c in CLASS_ORDER]
)
axes[1].set_title(f"{DESCRIPTIVE_LABEL} population distribution")
axes[1].set_ylabel("Weighted percent")
axes[1].tick_params(axis="x", rotation=20)
fig.suptitle("NFHS-5 anaemia severity: corrected outcome coding", fontweight="bold")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "01_corrected_target_distribution.png", bbox_inches="tight")
plt.show()


In [ ]:
# 4B. Weighted subgroup prevalence helper and final-domain EDA
def weighted_crosstab(frame, group_col, target_col="anaemia_level", weight_col="sample_weight"):
    work = frame[[group_col, target_col, weight_col]].dropna().copy()
    if work.empty:
        return pd.DataFrame()
    grouped = (
        work.groupby([group_col, target_col], observed=True)[weight_col]
        .sum().rename("weighted_n").reset_index()
    )
    grouped["weighted_percent"] = (
        100 * grouped["weighted_n"]
        / grouped.groupby(group_col, observed=True)["weighted_n"].transform("sum")
    )
    return grouped


eda_groups = [
    "age_group", "residence", "education_group", "wealth_quintile",
    "caste", "religion", "currently_pregnant", "currently_breastfeeding",
    "currently_amenorrheic", "pregnancy_termination_history",
    "health_insurance", "self_reported_diabetes", "diet_diversity_group",
    "bmi_group", "region_group",
]
eda_tables = {}
for group in eda_groups:
    if group in analysis_df and analysis_df[group].notna().any():
        table = weighted_crosstab(analysis_df, group)
        eda_tables[group] = table
        atomic_to_parquet(table, DIRS["tables"] / f"weighted_eda_{group}.parquet")

plot_groups = [
    "education_group", "wealth_quintile", "residence",
    "diet_diversity_group", "currently_breastfeeding", "region_group",
]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, group in zip(axes.flat, plot_groups):
    table = eda_tables.get(group, pd.DataFrame())
    if table.empty:
        ax.axis("off")
        continue
    pivot = table.pivot(
        index=group, columns="anaemia_level", values="weighted_percent"
    ).fillna(0).reindex(columns=CLASS_ORDER, fill_value=0)
    pivot.columns = [CLASS_LABELS[c] for c in CLASS_ORDER]
    pivot.plot(
        kind="bar", stacked=True, ax=ax,
        color=[CLASS_COLORS[c] for c in CLASS_ORDER], width=0.82,
    )
    ax.set_title(group.replace("_", " ").title())
    ax.set_ylabel(f"{DESCRIPTIVE_LABEL} percent")
    ax.tick_params(axis="x", rotation=25)
    ax.legend(fontsize=7)
fig.suptitle(
    f"{DESCRIPTIVE_LABEL} anaemia severity across pre-specified domains",
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "02_weighted_subgroups.png", bbox_inches="tight")
plt.show()


## 5. Pre-specified predictor sets and protected grouped splits

Three feature sets answer different questions:

1. **Policy (primary):** nutrition, reproductive health, access, caste,
   religion, state and BIMARU/Southern grouping.
2. **Without sensitive attributes:** removes caste and religion for a direct
   fairness/utility comparison while retaining geography.
3. **Portable:** removes state, caste, religion and region labels for
   state-held-out transportability analysis.

`sdist`, survey weights, PSU/strata identifiers, respondent identifiers,
`v456` and `v457` are never predictors. PSUs remain disjoint across the
70/10/20 training/calibration/locked-test split.


In [ ]:
# 5A. Final engineered feature sets
NUMERIC_BASE = [
    "age", "education_years", "children_ever_born", "births_last_5y",
    "age_first_birth", "ever_given_birth", "bmi",
    "barrier_permission", "barrier_money", "barrier_distance",
    "barrier_go_alone", "healthcare_barrier_count",
    "pulses_frequency", "leafy_vegetable_frequency", "fruit_frequency",
    "egg_frequency", "fish_frequency", "chicken_meat_frequency",
    "animal_source_food_score", "diet_diversity_score",
    "diet_weekly_or_daily_count", "underweight", "high_parity",
    "early_first_birth", "low_diet_diversity",
    "parity_x_underweight", "parity_x_current_pregnancy",
    "births5_x_underweight", "early_birth_x_parity",
    "births5_x_breastfeeding", "termination_x_high_parity",
    "underweight_x_low_diet_diversity",
]
CATEGORICAL_BASE = [
    "residence", "marital_status", "wealth_quintile", "currently_pregnant",
    "pregnancy_termination_history", "contraceptive_method",
    "currently_breastfeeding", "currently_amenorrheic", "health_insurance",
    "self_reported_diabetes", "drinking_water", "toilet_facility",
    "cooking_fuel",
]
POLICY_EXTRA = ["state_code", "caste", "religion", "region_group"]

FEATURE_SETS = {
    "policy": NUMERIC_BASE + CATEGORICAL_BASE + POLICY_EXTRA,
    "without_sensitive": NUMERIC_BASE + CATEGORICAL_BASE + ["state_code", "region_group"],
    "portable": NUMERIC_BASE + CATEGORICAL_BASE,
}

requested_features = FEATURE_SETS[PRIMARY_FEATURE_SET]
ACTIVE_FEATURES = [
    c for c in requested_features
    if c in analysis_df and analysis_df[c].notna().any()
]
DROPPED_ALL_MISSING_FEATURES = sorted(set(requested_features) - set(ACTIVE_FEATURES))
NUMERIC_FEATURES = [c for c in ACTIVE_FEATURES if c in NUMERIC_BASE]
CATEGORICAL_FEATURES = [c for c in ACTIVE_FEATURES if c not in NUMERIC_FEATURES]

assert_no_target_leakage(ACTIVE_FEATURES)
assert "raw_v457" not in ACTIVE_FEATURES
assert "adjusted_hb_gdl" not in ACTIVE_FEATURES
assert not (
    set(ACTIVE_FEATURES)
    & {"sample_weight", "psu", "strata", "row_id", "respondent_key", "district_code"}
)

feature_schema = {
    "raw_schema": FINAL_RAW_COLUMNS,
    "primary_feature_set": PRIMARY_FEATURE_SET,
    "feature_sets": FEATURE_SETS,
    "active_features": ACTIVE_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "dropped_all_missing": DROPPED_ALL_MISSING_FEATURES,
    "outcome_derived_exclusions": sorted(OUTCOME_DERIVED_COLUMNS),
    "survey_design_available": SURVEY_DESIGN_AVAILABLE,
    "paper_contract_passed": contract_report["paper_contract_passed"],
    "interaction_features": [
        c for c in ACTIVE_FEATURES if "_x_" in c
    ],
}
atomic_write_json(feature_schema, DIRS["reports"] / "feature_schema.json")

print("Active features:", len(ACTIVE_FEATURES))
print("Numeric:", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
print("Dropped because entirely missing:", DROPPED_ALL_MISSING_FEATURES)


In [ ]:
# 5B. Group- and geography-aware 70/10/20 split
def choose_best_stratified_group_fold(y, groups, state, n_splits, random_state):
    y = pd.Series(np.asarray(y)).reset_index(drop=True)
    groups = pd.Series(np.asarray(groups)).reset_index(drop=True)
    state = pd.Series(np.asarray(state)).reset_index(drop=True)
    composite = state.fillna(-1).astype(str) + "|" + y.astype(str)
    composite_counts = composite.value_counts()
    if (composite_counts < n_splits).any():
        split_target = y
    else:
        split_target = composite

    splitter = StratifiedGroupKFold(
        n_splits=n_splits, shuffle=True, random_state=random_state
    )
    overall = y.value_counts(normalize=True).reindex(CLASS_ORDER, fill_value=0).values
    candidates = []
    dummy_x = np.zeros((len(y), 1))
    for fold, (train_pos, valid_pos) in enumerate(
        splitter.split(dummy_x, split_target, groups)
    ):
        fold_dist = (
            y.iloc[valid_pos].value_counts(normalize=True)
            .reindex(CLASS_ORDER, fill_value=0).values
        )
        score = float(np.abs(fold_dist - overall).sum())
        candidates.append((score, fold, train_pos, valid_pos))
    return min(candidates, key=lambda item: (item[0], item[1]))


split_file = DIRS["cache"] / "protected_splits.npz"
split_params = {
    "test_fraction": TEST_FRACTION,
    "calibration_fraction": CALIBRATION_FRACTION,
    "random_state": SPLIT_RANDOM_STATE,
    "splitter": "StratifiedGroupKFold_by_state_composite_PSU_v4_post_pilot",
}

if stage_is_valid("splits", [split_file], split_params):
    split_npz = np.load(split_file)
    TRAIN_POS = split_npz["train_pos"]
    CALIB_POS = split_npz["calibration_pos"]
    TEST_POS = split_npz["test_pos"]
    print("Loaded protected split checkpoint.")
else:
    y_all = analysis_df["anaemia_level"].reset_index(drop=True)
    groups_all = analysis_df["psu"].fillna(analysis_df["row_id"]).reset_index(drop=True)
    state_all = analysis_df["state_code"].reset_index(drop=True)

    test_splits = max(2, round(1 / TEST_FRACTION))
    _, _, dev_pos, TEST_POS = choose_best_stratified_group_fold(
        y_all, groups_all, state_all, test_splits, SPLIT_RANDOM_STATE
    )

    y_dev = y_all.iloc[dev_pos].reset_index(drop=True)
    groups_dev = groups_all.iloc[dev_pos].reset_index(drop=True)
    state_dev = state_all.iloc[dev_pos].reset_index(drop=True)
    calib_share_within_dev = CALIBRATION_FRACTION / (1 - TEST_FRACTION)
    calib_splits = max(2, round(1 / calib_share_within_dev))
    _, _, train_within_dev, calib_within_dev = choose_best_stratified_group_fold(
        y_dev, groups_dev, state_dev, calib_splits, SPLIT_RANDOM_STATE + 1
    )
    TRAIN_POS = np.asarray(dev_pos)[train_within_dev]
    CALIB_POS = np.asarray(dev_pos)[calib_within_dev]
    TEST_POS = np.asarray(TEST_POS)

    atomic_savez(
        split_file,
        train_pos=TRAIN_POS,
        calibration_pos=CALIB_POS,
        test_pos=TEST_POS,
    )
    mark_stage_complete(
        "splits", [split_file], split_params,
        extra={
            "train_n": len(TRAIN_POS),
            "calibration_n": len(CALIB_POS),
            "test_n": len(TEST_POS),
        },
    )

def group_set(pos):
    return set(analysis_df.iloc[pos]["psu"].dropna().tolist())

assert group_set(TRAIN_POS).isdisjoint(group_set(CALIB_POS))
assert group_set(TRAIN_POS).isdisjoint(group_set(TEST_POS))
assert group_set(CALIB_POS).isdisjoint(group_set(TEST_POS))
assert len(set(TRAIN_POS) & set(CALIB_POS)) == 0
assert len(set(TRAIN_POS) & set(TEST_POS)) == 0
assert len(set(CALIB_POS) & set(TEST_POS)) == 0

X = analysis_df[ACTIVE_FEATURES].copy()
y = analysis_df["anaemia_level"].astype(int)
weights = analysis_df["sample_weight"].astype(float)
groups = analysis_df["psu"]
states = analysis_df["state_code"]

X_train, y_train = X.iloc[TRAIN_POS], y.iloc[TRAIN_POS]
X_calib, y_calib = X.iloc[CALIB_POS], y.iloc[CALIB_POS]
X_test, y_test = X.iloc[TEST_POS], y.iloc[TEST_POS]
w_train, w_calib, w_test = weights.iloc[TRAIN_POS], weights.iloc[CALIB_POS], weights.iloc[TEST_POS]
g_train, g_calib, g_test = groups.iloc[TRAIN_POS], groups.iloc[CALIB_POS], groups.iloc[TEST_POS]

split_summary = []
for name, pos in [("train", TRAIN_POS), ("calibration", CALIB_POS), ("test", TEST_POS)]:
    subset = analysis_df.iloc[pos]
    row = {
        "split": name,
        "n": len(subset),
        "n_psu": subset["psu"].nunique(),
        "n_strata": subset["strata"].nunique(),
        "n_state_district_pairs": subset[["state_code", "district_code"]].drop_duplicates().shape[0],
    }
    assert set(subset["anaemia_level"].unique()) == set(CLASS_ORDER)
    for c in CLASS_ORDER:
        row[f"class_{c}_pct"] = 100 * (subset["anaemia_level"] == c).mean()
        row[f"weighted_class_{c}_pct"] = 100 * weighted_class_prevalence(
            subset["anaemia_level"], subset["sample_weight"]
        )[c]
    split_summary.append(row)
split_summary = pd.DataFrame(split_summary)
display(split_summary.round(3))
atomic_to_parquet(split_summary, DIRS["tables"] / "split_summary.parquet")


# 6A. Metrics and calibration utilities
def multiclass_brier(y_true, proba, sample_weight=None):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    onehot = np.eye(len(CLASS_ORDER))[y_true]
    per_row = np.sum((onehot - proba) ** 2, axis=1)
    return float(np.average(per_row, weights=sample_weight))


def multiclass_ece(y_true, proba, n_bins=15, sample_weight=None):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    pred = proba.argmax(axis=1)
    confidence = proba.max(axis=1)
    correct = (pred == y_true).astype(float)
    weights_local = (
        np.ones(len(y_true), dtype=float)
        if sample_weight is None else np.asarray(sample_weight, dtype=float)
    )
    edges = np.linspace(0, 1, n_bins + 1)
    total = weights_local.sum()
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (
            confidence <= hi if hi == 1 else confidence < hi
        )
        if not mask.any():
            continue
        mass = weights_local[mask].sum() / total
        acc = np.average(correct[mask], weights=weights_local[mask])
        conf = np.average(confidence[mask], weights=weights_local[mask])
        ece += mass * abs(acc - conf)
    return float(ece)


def safe_macro_auc(y_true, proba, sample_weight=None):
    y_bin = label_binarize(y_true, classes=CLASS_ORDER)
    try:
        return float(roc_auc_score(
            y_bin, proba, average="macro", multi_class="ovr",
            sample_weight=sample_weight
        ))
    except ValueError:
        return np.nan


def safe_macro_auprc(y_true, proba, sample_weight=None):
    y_bin = label_binarize(y_true, classes=CLASS_ORDER)
    try:
        return float(average_precision_score(
            y_bin, proba, average="macro", sample_weight=sample_weight
        ))
    except ValueError:
        return np.nan


def metric_bundle(y_true, proba, sample_weight=None, prefix=""):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    # Defensive clipping and re-normalization to ensure probabilities sum to 1.
    proba = np.clip(proba, 1e-15, 1 - 1e-15)
    proba = proba / proba.sum(axis=1, keepdims=True)

    pred = proba.argmax(axis=1)
    sw = None if sample_weight is None else normalized_weights(sample_weight)
    result = {
        f"{prefix}accuracy": accuracy_score(y_true, pred, sample_weight=sw),
        f"{prefix}balanced_accuracy": balanced_accuracy_score(y_true, pred, sample_weight=sw),
        f"{prefix}macro_f1": f1_score(y_true, pred, average="macro", sample_weight=sw, zero_division=0),
        f"{prefix}weighted_f1": f1_score(y_true, pred, average="weighted", sample_weight=sw, zero_division=0),
        f"{prefix}macro_auroc": safe_macro_auc(y_true, proba, sw),
        f"{prefix}macro_auprc": safe_macro_auprc(y_true, proba, sw),
        f"{prefix}log_loss": log_loss(y_true, proba, labels=CLASS_ORDER, sample_weight=sw),
        f"{prefix}brier": multiclass_brier(y_true, proba, sw),
        f"{prefix}ece": multiclass_ece(y_true, proba, sample_weight=sw),
        f"{prefix}kappa": cohen_kappa_score(y_true, pred, sample_weight=sw),
        f"{prefix}quadratic_kappa": cohen_kappa_score(
            y_true, pred, weights="quadratic", sample_weight=sw
        ),
        f"{prefix}ordinal_mae": mean_absolute_error(y_true, pred, sample_weight=sw),
    }
    for c in CLASS_ORDER:
        result[f"{prefix}recall_class_{c}"] = recall_score(
            y_true, pred, labels=[c], average="macro",
            sample_weight=sw, zero_division=0
        )
        result[f"{prefix}precision_class_{c}"] = precision_score(
            y_true, pred, labels=[c], average="macro",
            sample_weight=sw, zero_division=0
        )
        binary = (y_true == c).astype(int)
        try:
            result[f"{prefix}auprc_class_{c}"] = average_precision_score(
                binary, proba[:, c], sample_weight=sw
            )
            result[f"{prefix}auroc_class_{c}"] = roc_auc_score(
                binary, proba[:, c], sample_weight=sw
            )
        except ValueError:
            result[f"{prefix}auprc_class_{c}"] = np.nan
            result[f"{prefix}auroc_class_{c}"] = np.nan
    return {k: float(v) if pd.notna(v) else np.nan for k, v in result.items()}


class TemperatureScaler:
    def __init__(self, temperature=1.0):
        self.temperature = float(temperature)

    @staticmethod
    def _softmax(logits):
        logits = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(logits)
        return exp / exp.sum(axis=1, keepdims=True)

    def fit(self, proba, y, sample_weight=None):
        proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
        logits = np.log(proba)
        y = np.asarray(y, dtype=int)
        sw = None if sample_weight is None else normalized_weights(sample_weight)

        def objective(log_t):
            t = np.exp(log_t)
            calibrated = self._softmax(logits / t)
            return log_loss(y, calibrated, labels=CLASS_ORDER, sample_weight=sw)

        result = minimize_scalar(objective, bounds=(-3.0, 3.0), method="bounded")
        self.temperature = float(np.exp(result.x))
        return self

    def transform(self, proba):
        proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
        return self._softmax(np.log(proba) / self.temperature)

In [ ]:
# 6A. Metrics and calibration utilities
def normalize_probabilities(proba):
    """Return finite float64 row-normalised multiclass probabilities."""
    values = np.asarray(proba, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(CLASS_ORDER):
        raise ValueError(f"Expected an (n, {len(CLASS_ORDER)}) probability matrix")
    if not np.isfinite(values).all():
        raise ValueError("Probability matrix contains NaN or infinite values")
    values = np.clip(values, 1e-12, 1.0)
    totals = values.sum(axis=1, keepdims=True)
    if np.any(totals <= 0):
        raise ValueError("Probability matrix contains a non-positive row sum")
    return values / totals


def multiclass_brier(y_true, proba, sample_weight=None):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    onehot = np.eye(len(CLASS_ORDER))[y_true]
    per_row = np.sum((onehot - proba) ** 2, axis=1)
    return float(np.average(per_row, weights=sample_weight))


def multiclass_ece(y_true, proba, n_bins=15, sample_weight=None):
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=float)
    pred = proba.argmax(axis=1)
    confidence = proba.max(axis=1)
    correct = (pred == y_true).astype(float)
    weights_local = (
        np.ones(len(y_true), dtype=float)
        if sample_weight is None else np.asarray(sample_weight, dtype=float)
    )
    edges = np.linspace(0, 1, n_bins + 1)
    total = weights_local.sum()
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence >= lo) & (
            confidence <= hi if hi == 1 else confidence < hi
        )
        if not mask.any():
            continue
        mass = weights_local[mask].sum() / total
        acc = np.average(correct[mask], weights=weights_local[mask])
        conf = np.average(confidence[mask], weights=weights_local[mask])
        ece += mass * abs(acc - conf)
    return float(ece)


def safe_macro_auc(y_true, proba, sample_weight=None):
    y_bin = label_binarize(y_true, classes=CLASS_ORDER)
    try:
        return float(roc_auc_score(
            y_bin, proba, average="macro", multi_class="ovr",
            sample_weight=sample_weight
        ))
    except ValueError:
        return np.nan


def safe_macro_auprc(y_true, proba, sample_weight=None):
    y_bin = label_binarize(y_true, classes=CLASS_ORDER)
    try:
        return float(average_precision_score(
            y_bin, proba, average="macro", sample_weight=sample_weight
        ))
    except ValueError:
        return np.nan


def metric_bundle(y_true, proba, sample_weight=None, prefix=""):
    y_true = np.asarray(y_true, dtype=int)
    proba = normalize_probabilities(proba)
    pred = proba.argmax(axis=1)
    sw = None if sample_weight is None else normalized_weights(sample_weight)
    result = {
        f"{prefix}accuracy": accuracy_score(y_true, pred, sample_weight=sw),
        f"{prefix}balanced_accuracy": balanced_accuracy_score(y_true, pred, sample_weight=sw),
        f"{prefix}macro_f1": f1_score(y_true, pred, average="macro", sample_weight=sw, zero_division=0),
        f"{prefix}weighted_f1": f1_score(y_true, pred, average="weighted", sample_weight=sw, zero_division=0),
        f"{prefix}macro_auroc": safe_macro_auc(y_true, proba, sw),
        f"{prefix}macro_auprc": safe_macro_auprc(y_true, proba, sw),
        f"{prefix}log_loss": log_loss(y_true, proba, labels=CLASS_ORDER, sample_weight=sw),
        f"{prefix}brier": multiclass_brier(y_true, proba, sw),
        f"{prefix}ece": multiclass_ece(y_true, proba, sample_weight=sw),
        f"{prefix}kappa": cohen_kappa_score(y_true, pred, sample_weight=sw),
        f"{prefix}quadratic_kappa": cohen_kappa_score(
            y_true, pred, weights="quadratic", sample_weight=sw
        ),
        f"{prefix}ordinal_mae": mean_absolute_error(y_true, pred, sample_weight=sw),
    }
    for c in CLASS_ORDER:
        result[f"{prefix}recall_class_{c}"] = recall_score(
            y_true, pred, labels=[c], average="macro",
            sample_weight=sw, zero_division=0
        )
        result[f"{prefix}precision_class_{c}"] = precision_score(
            y_true, pred, labels=[c], average="macro",
            sample_weight=sw, zero_division=0
        )
        binary = (y_true == c).astype(int)
        try:
            result[f"{prefix}auprc_class_{c}"] = average_precision_score(
                binary, proba[:, c], sample_weight=sw
            )
            result[f"{prefix}auroc_class_{c}"] = roc_auc_score(
                binary, proba[:, c], sample_weight=sw
            )
        except ValueError:
            result[f"{prefix}auprc_class_{c}"] = np.nan
            result[f"{prefix}auroc_class_{c}"] = np.nan
    return {k: float(v) if pd.notna(v) else np.nan for k, v in result.items()}


class TemperatureScaler:
    def __init__(self, temperature=1.0):
        self.temperature = float(temperature)

    @staticmethod
    def _softmax(logits):
        logits = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(logits)
        return exp / exp.sum(axis=1, keepdims=True)

    def fit(self, proba, y, sample_weight=None):
        proba = normalize_probabilities(proba)
        logits = np.log(proba)
        y = np.asarray(y, dtype=int)
        sw = None if sample_weight is None else normalized_weights(sample_weight)

        def objective(log_t):
            t = np.exp(log_t)
            calibrated = self._softmax(logits / t)
            return log_loss(y, calibrated, labels=CLASS_ORDER, sample_weight=sw)

        result = minimize_scalar(objective, bounds=(-3.0, 3.0), method="bounded")
        self.temperature = float(np.exp(result.x))
        return self

    def transform(self, proba):
        proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
        return self._softmax(np.log(proba) / self.temperature)


In [ ]:
# 6B. Ordinal classifier and preprocessing factory
class OrdinalThresholdClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, C=1.0, max_iter=500, class_weight="balanced", random_state=42):
        self.C = C
        self.max_iter = max_iter
        self.class_weight = class_weight
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        y = np.asarray(y, dtype=int)
        self.classes_ = np.array(CLASS_ORDER)
        self.models_ = []
        for threshold in range(len(CLASS_ORDER) - 1):
            binary_y = (y > threshold).astype(int)
            model = LogisticRegression(
                C=self.C,
                max_iter=self.max_iter,
                solver="saga",
                class_weight=self.class_weight,
                random_state=self.random_state,
            )
            model.fit(X, binary_y, sample_weight=sample_weight)
            self.models_.append(model)
        return self

    def predict_proba(self, X):
        cumulative = np.column_stack([
            model.predict_proba(X)[:, 1] for model in self.models_
        ])
        # Enforce P(Y>0) >= P(Y>1) >= P(Y>2).
        cumulative = np.minimum.accumulate(cumulative, axis=1)
        proba = np.column_stack([
            1 - cumulative[:, 0],
            cumulative[:, 0] - cumulative[:, 1],
            cumulative[:, 1] - cumulative[:, 2],
            cumulative[:, 2],
        ])
        proba = np.clip(proba, 0, 1)
        return proba / proba.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def make_preprocessor():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median", keep_empty_features=True, add_indicator=True
        )),
        ("scaler", StandardScaler(with_mean=False)),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(
            strategy="most_frequent", keep_empty_features=True
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=20,
            sparse_output=True,
        )),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, NUMERIC_FEATURES),
            ("cat", categorical_pipe, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def make_estimator(model_name, params=None):
    params = dict(params or {})
    if model_name == "dummy":
        return DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
    if model_name == "ordinal_logit":
        return OrdinalThresholdClassifier(
            C=params.get("C", 1.0),
            max_iter=600,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    if model_name == "multinomial_logit":
        return LogisticRegression(
            C=params.get("C", 1.0),
            max_iter=600,
            solver="saga",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    if model_name == "random_forest":
        defaults = {
            "n_estimators": 350,
            "max_depth": 18,
            "min_samples_split": 5,
            "min_samples_leaf": 3,
            "max_features": "sqrt",
        }
        defaults.update(params)
        return RandomForestClassifier(
            **defaults,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
    if model_name == "lightgbm":
        defaults = {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "num_leaves": 63,
            "max_depth": -1,
            "min_child_samples": 40,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
        }
        defaults.update(params)
        return LGBMClassifier(
            **defaults,
            objective="multiclass",
            num_class=4,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbosity=-1,
        )
    if model_name == "xgboost":
        defaults = {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 7,
            "min_child_weight": 5,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "gamma": 0.0,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
        }
        defaults.update(params)
        gpu_params = {"device": "cuda"} if USE_GPU else {"tree_method": "hist"}
        return XGBClassifier(
            **defaults,
            **gpu_params,
            objective="multi:softprob",
            num_class=4,
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )
    if model_name == "catboost":
        defaults = {
            "iterations": 500,
            "depth": 7,
            "learning_rate": 0.05,
            "l2_leaf_reg": 3.0,
            "random_strength": 1.0,
            "bagging_temperature": 0.5,
        }
        defaults.update(params)
        return CatBoostClassifier(
            **defaults,
            loss_function="MultiClass",
            eval_metric="TotalF1:average=Macro",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
            task_type="GPU" if USE_GPU else "CPU",
            thread_count=N_JOBS,
        )
    raise KeyError(f"Unknown model: {model_name}")


def make_pipeline(model_name, params=None):
    return Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", make_estimator(model_name, params)),
    ])


def fit_kwargs_for_model(model_name, y_fit, survey_weights_fit=None):
    if model_name == "xgboost":
        class_w = compute_sample_weight(class_weight="balanced", y=y_fit)
        if survey_weights_fit is not None:
            class_w = class_w * normalized_weights(survey_weights_fit)
        return {"model__sample_weight": class_w}
    if model_name in {"dummy", "ordinal_logit", "multinomial_logit", "random_forest", "lightgbm"}:
        if survey_weights_fit is not None:
            return {"model__sample_weight": normalized_weights(survey_weights_fit)}
    return {}


# Minimal self-tests independent of the full modelling run.
test_map = pd.Series([4, 3, 2, 1]).map(TARGET_MAP_DHS_1_TO_4).tolist()
assert test_map == [0, 1, 2, 3]
assert_no_target_leakage(ACTIVE_FEATURES)
print("Core leakage, mapping and model-factory self-tests passed.")


## 7. Persistent Optuna hyperparameter optimisation

The Optuna SQLite database is stored inside the run fingerprint. Each completed trial survives a runtime reset. Increasing `CFG["hpo_trials"]` adds only the remaining trials.

HPO uses a PSU-grouped subset and grouped cross-validation. Preprocessing is fitted separately inside each fold; no global scaling, imputation or resampling occurs before cross-validation.


In [ ]:
# 7A. Group-sampling, grouped-CV and search spaces
def group_sample_positions(groups_local, max_rows, random_state=42):
    groups_series = pd.Series(np.asarray(groups_local)).reset_index(drop=True)
    if len(groups_series) <= max_rows:
        return np.arange(len(groups_series))
    rng = np.random.default_rng(random_state)
    unique_groups = groups_series.dropna().unique()
    rng.shuffle(unique_groups)
    counts_local = groups_series.value_counts()
    selected, running = [], 0
    for group in unique_groups:
        selected.append(group)
        running += int(counts_local.get(group, 0))
        if running >= max_rows:
            break
    mask = groups_series.isin(selected).to_numpy()
    return np.flatnonzero(mask)


HPO_POS = group_sample_positions(g_train, CFG["hpo_rows"], RANDOM_STATE)
X_hpo = X_train.iloc[HPO_POS]
y_hpo = y_train.iloc[HPO_POS]
g_hpo = g_train.iloc[HPO_POS]
w_hpo = w_train.iloc[HPO_POS]
print("HPO sample:", X_hpo.shape, "| PSUs:", g_hpo.nunique())


def suggest_params(trial, model_name):
    if model_name == "random_forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 650, step=50),
            "max_depth": trial.suggest_int("max_depth", 8, 28),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 15),
            "max_features": trial.suggest_categorical(
                "max_features", ["sqrt", "log2", 0.5]
            ),
        }
    if model_name == "lightgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 250, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 160),
            "max_depth": trial.suggest_int("max_depth", 5, 14),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 120),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }
    if model_name == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 250, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 11),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 15.0),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }
    if model_name == "catboost":
        return {
            "iterations": trial.suggest_int("iterations", 250, 750, step=50),
            "depth": trial.suggest_int("depth", 5, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.15, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 20.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        }
    raise KeyError(model_name)


def grouped_cv_score(model_name, params, X_local, y_local, groups_local, weights_local, n_splits):
    splitter = StratifiedGroupKFold(
        n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE
    )
    fold_scores = []
    for fold, (tr, va) in enumerate(
        splitter.split(X_local, y_local, groups_local)
    ):
        pipeline = make_pipeline(model_name, params)
        fit_kwargs = fit_kwargs_for_model(
            model_name, y_local.iloc[tr], None
        )
        pipeline.fit(X_local.iloc[tr], y_local.iloc[tr], **fit_kwargs)
        proba = pipeline.predict_proba(X_local.iloc[va])
        fold_scores.append(
            f1_score(
                y_local.iloc[va], proba.argmax(axis=1),
                average="macro", zero_division=0
            )
        )
        del pipeline
    return float(np.mean(fold_scores)), fold_scores


def run_or_resume_optuna(model_name, n_trials=None, suffix="primary", data=None):
    n_trials = int(n_trials or CFG["hpo_trials"])
    X_local, y_local, g_local, w_local = data or (X_hpo, y_hpo, g_hpo, w_hpo)
    study_name = f"{PIPELINE_VERSION}_{RUN_FINGERPRINT}_{model_name}_{suffix}"
    study = optuna.create_study(
        study_name=study_name,
        storage=OPTUNA_DB,
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        load_if_exists=True,
    )
    completed = sum(
        trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials
    )
    remaining = max(0, n_trials - completed)

    def objective(trial):
        params = suggest_params(trial, model_name)
        score, fold_scores = grouped_cv_score(
            model_name, params, X_local, y_local, g_local, w_local,
            n_splits=min(CFG["cv_splits"], 5),
        )
        for i, fold_score in enumerate(fold_scores):
            trial.set_user_attr(f"fold_{i}_macro_f1", float(fold_score))
        return score

    if remaining:
        print(f"{model_name}: continuing {remaining} Optuna trials ({completed} already complete)")
        study.optimize(
            objective,
            n_trials=remaining,
            gc_after_trial=True,
            show_progress_bar=True,
            catch=(MemoryError,),
        )
    else:
        print(f"{model_name}: all {completed} requested trials already complete")

    result = {
        "model": model_name,
        "study_name": study_name,
        "completed_trials": sum(
            t.state == optuna.trial.TrialState.COMPLETE for t in study.trials
        ),
        "best_value": float(study.best_value),
        "best_params": study.best_params,
    }
    atomic_write_json(result, DIRS["studies"] / f"{model_name}_{suffix}_best.json")
    return result


BEST_PARAMS = {
    "dummy": {},
    "ordinal_logit": {},
    "multinomial_logit": {},
}


In [ ]:
# 7.random_forest: resumable HPO
if "random_forest" in ENABLED_MODELS and "random_forest" in HPO_MODELS:
    hpo_result_random_forest = run_or_resume_optuna("random_forest")
    BEST_PARAMS["random_forest"] = hpo_result_random_forest["best_params"]
    print(json.dumps(hpo_result_random_forest, indent=2))
else:
    BEST_PARAMS["random_forest"] = {}


In [ ]:
# 7.lightgbm: resumable HPO
if "lightgbm" in ENABLED_MODELS and "lightgbm" in HPO_MODELS:
    hpo_result_lightgbm = run_or_resume_optuna("lightgbm")
    BEST_PARAMS["lightgbm"] = hpo_result_lightgbm["best_params"]
    print(json.dumps(hpo_result_lightgbm, indent=2))
else:
    BEST_PARAMS["lightgbm"] = {}


In [ ]:
# 7.xgboost: resumable HPO
if "xgboost" in ENABLED_MODELS and "xgboost" in HPO_MODELS:
    hpo_result_xgboost = run_or_resume_optuna("xgboost")
    BEST_PARAMS["xgboost"] = hpo_result_xgboost["best_params"]
    print(json.dumps(hpo_result_xgboost, indent=2))
else:
    BEST_PARAMS["xgboost"] = {}


In [ ]:
# 7.catboost: resumable HPO
if "catboost" in ENABLED_MODELS and "catboost" in HPO_MODELS:
    hpo_result_catboost = run_or_resume_optuna("catboost")
    BEST_PARAMS["catboost"] = hpo_result_catboost["best_params"]
    print(json.dumps(hpo_result_catboost, indent=2))
else:
    BEST_PARAMS["catboost"] = {}


## 8. Per-model grouped validation and full training checkpoints

Each model has its own checkpoint containing:

- the fitted preprocessing-plus-model pipeline,
- grouped-CV fold predictions and metrics,
- the exact hyperparameters and feature-schema signature.

If a runtime fails while training one model, previously completed models remain reusable.


In [ ]:
# 8A. Checkpointed grouped CV and full-training routine
def train_model_checkpoint(model_name, params):
    stage = f"model_{model_name}"
    model_path = DIRS["models"] / f"{model_name}.joblib"
    cv_path = DIRS["tables"] / f"cv_{model_name}.parquet"
    oof_path = DIRS["predictions"] / f"oof_{model_name}.npz"
    parameters = {
        "model_name": model_name,
        "params": params,
        "features": ACTIVE_FEATURES,
        "cv_splits": CFG["cv_splits"],
        "training_weights": "class_only_primary",
    }
    outputs = [model_path, cv_path, oof_path]
    if stage_is_valid(stage, outputs, parameters):
        model = joblib.load(model_path)
        cv_frame = pd.read_parquet(cv_path)
        oof = np.load(oof_path)["proba"]
        print(f"{model_name}: loaded compatible model/CV checkpoint")
        return model, cv_frame, oof

    splitter = StratifiedGroupKFold(
        n_splits=CFG["cv_splits"], shuffle=True, random_state=RANDOM_STATE
    )
    oof = np.full((len(X_train), 4), np.nan, dtype=np.float32)
    fold_rows = []
    for fold, (tr, va) in enumerate(splitter.split(X_train, y_train, g_train)):
        fold_path = DIRS["models"] / f"{model_name}_cv_fold_{fold}.joblib"
        fold_pred_path = DIRS["predictions"] / f"{model_name}_cv_fold_{fold}.npz"
        fold_stage = f"{stage}_fold_{fold}"
        fold_parameters = {**parameters, "fold": fold}
        if stage_is_valid(fold_stage, [fold_path, fold_pred_path], fold_parameters):
            fold_model = joblib.load(fold_path)
            fold_proba = np.load(fold_pred_path)["proba"]
            print(f"{model_name} fold {fold}: loaded")
        else:
            fold_model = make_pipeline(model_name, params)
            fit_kwargs = fit_kwargs_for_model(
                model_name, y_train.iloc[tr], None
            )
            t0 = time.time()
            fold_model.fit(X_train.iloc[tr], y_train.iloc[tr], **fit_kwargs)
            fold_proba = fold_model.predict_proba(X_train.iloc[va]).astype(np.float32)
            atomic_joblib_dump(fold_model, fold_path)
            atomic_savez(fold_pred_path, proba=fold_proba, valid_pos=va)
            mark_stage_complete(
                fold_stage, [fold_path, fold_pred_path], fold_parameters,
                extra={"seconds": time.time() - t0}
            )
            print(f"{model_name} fold {fold}: trained and saved")

        oof[va] = fold_proba
        fold_metric = metric_bundle(y_train.iloc[va], fold_proba)
        fold_metric.update({"model": model_name, "fold": fold, "n": len(va)})
        fold_rows.append(fold_metric)
        del fold_model

    if np.isnan(oof).any():
        raise RuntimeError(f"OOF predictions incomplete for {model_name}")
    cv_frame = pd.DataFrame(fold_rows)

    full_model = make_pipeline(model_name, params)
    full_fit_kwargs = fit_kwargs_for_model(model_name, y_train, None)
    full_model.fit(X_train, y_train, **full_fit_kwargs)

    atomic_joblib_dump(full_model, model_path)
    atomic_to_parquet(cv_frame, cv_path, index=False)
    atomic_savez(oof_path, proba=oof)
    mark_stage_complete(
        stage, outputs, parameters,
        extra={
            "mean_macro_f1": float(cv_frame["macro_f1"].mean()),
            "std_macro_f1": float(cv_frame["macro_f1"].std(ddof=1)),
        },
    )
    return full_model, cv_frame, oof


TRAINED_MODELS = {}
CV_TABLES = {}
OOF_PREDICTIONS = {}


In [ ]:
# 8.dummy: grouped CV + full fit, independently restartable
if "dummy" in ENABLED_MODELS:
    model_dummy, cv_dummy, oof_dummy = train_model_checkpoint(
        "dummy", BEST_PARAMS.get("dummy", {})
    )
    TRAINED_MODELS["dummy"] = model_dummy
    CV_TABLES["dummy"] = cv_dummy
    OOF_PREDICTIONS["dummy"] = oof_dummy


In [ ]:
# 8.ordinal_logit: grouped CV + full fit, independently restartable
if "ordinal_logit" in ENABLED_MODELS:
    model_ordinal_logit, cv_ordinal_logit, oof_ordinal_logit = train_model_checkpoint(
        "ordinal_logit", BEST_PARAMS.get("ordinal_logit", {})
    )
    TRAINED_MODELS["ordinal_logit"] = model_ordinal_logit
    CV_TABLES["ordinal_logit"] = cv_ordinal_logit
    OOF_PREDICTIONS["ordinal_logit"] = oof_ordinal_logit


In [ ]:
# 8.multinomial_logit: grouped CV + full fit, independently restartable
if "multinomial_logit" in ENABLED_MODELS:
    model_multinomial_logit, cv_multinomial_logit, oof_multinomial_logit = train_model_checkpoint(
        "multinomial_logit", BEST_PARAMS.get("multinomial_logit", {})
    )
    TRAINED_MODELS["multinomial_logit"] = model_multinomial_logit
    CV_TABLES["multinomial_logit"] = cv_multinomial_logit
    OOF_PREDICTIONS["multinomial_logit"] = oof_multinomial_logit


In [ ]:
# 8.random_forest: grouped CV + full fit, independently restartable
if "random_forest" in ENABLED_MODELS:
    model_random_forest, cv_random_forest, oof_random_forest = train_model_checkpoint(
        "random_forest", BEST_PARAMS.get("random_forest", {})
    )
    TRAINED_MODELS["random_forest"] = model_random_forest
    CV_TABLES["random_forest"] = cv_random_forest
    OOF_PREDICTIONS["random_forest"] = oof_random_forest


In [ ]:
# 8.lightgbm: grouped CV + full fit, independently restartable
if "lightgbm" in ENABLED_MODELS:
    model_lightgbm, cv_lightgbm, oof_lightgbm = train_model_checkpoint(
        "lightgbm", BEST_PARAMS.get("lightgbm", {})
    )
    TRAINED_MODELS["lightgbm"] = model_lightgbm
    CV_TABLES["lightgbm"] = cv_lightgbm
    OOF_PREDICTIONS["lightgbm"] = oof_lightgbm


In [ ]:
# 8.xgboost: grouped CV + full fit, independently restartable
if "xgboost" in ENABLED_MODELS:
    model_xgboost, cv_xgboost, oof_xgboost = train_model_checkpoint(
        "xgboost", BEST_PARAMS.get("xgboost", {})
    )
    TRAINED_MODELS["xgboost"] = model_xgboost
    CV_TABLES["xgboost"] = cv_xgboost
    OOF_PREDICTIONS["xgboost"] = oof_xgboost


In [ ]:
# 8.catboost: grouped CV + full fit, independently restartable
if "catboost" in ENABLED_MODELS:
    model_catboost, cv_catboost, oof_catboost = train_model_checkpoint(
        "catboost", BEST_PARAMS.get("catboost", {})
    )
    TRAINED_MODELS["catboost"] = model_catboost
    CV_TABLES["catboost"] = cv_catboost
    OOF_PREDICTIONS["catboost"] = oof_catboost


In [ ]:
# 8I. Development-CV model ranking (test set remains untouched)
cv_summary_rows = []
for name, frame in CV_TABLES.items():
    row = {
        "model": name,
        "mean_macro_f1": frame["macro_f1"].mean(),
        "sd_macro_f1": frame["macro_f1"].std(ddof=1),
        "mean_macro_auroc": frame["macro_auroc"].mean(),
        "mean_macro_auprc": frame["macro_auprc"].mean(),
        "mean_severe_recall": frame["recall_class_3"].mean(),
        "mean_severe_auprc": frame["auprc_class_3"].mean(),
        "mean_log_loss": frame["log_loss"].mean(),
    }
    cv_summary_rows.append(row)

cv_summary = (
    pd.DataFrame(cv_summary_rows)
    .sort_values(["mean_macro_f1", "mean_severe_auprc"], ascending=False)
    .reset_index(drop=True)
)
display(cv_summary.round(4))
atomic_to_parquet(cv_summary, DIRS["tables"] / "cv_model_summary.parquet")

TOP_MODEL_NAMES = cv_summary.loc[
    cv_summary["model"] != "dummy", "model"
].head(2).tolist()
if not TOP_MODEL_NAMES:
    raise RuntimeError("No non-dummy model completed.")
print("Top models selected using grouped development CV only:", TOP_MODEL_NAMES)


## 9. Probability calibration and pre-specified ensemble

Class balancing changes the apparent class priors, so raw probabilities may be miscalibrated. Temperature scaling is fitted only on the separate PSU-disjoint calibration set. The top two candidates were selected using development CV—not the test set.

An ensemble is retained only if it improves survey-weighted calibration-set log loss. The selection rule is fixed before locked-test evaluation.


In [ ]:
# 9A. Cache calibration/test probabilities for frozen base models
def cached_model_probabilities(model_name, model, split_name, X_split):
    path = DIRS["predictions"] / f"{model_name}_{split_name}_proba.npz"
    stage = f"predict_{model_name}_{split_name}"
    parameters = {
        "model_name": model_name,
        "split": split_name,
        "probability_policy": PROBABILITY_POLICY_VERSION,
        "model_signature": stage_signature(
            f"model_{model_name}",
            {
                "model_name": model_name,
                "params": BEST_PARAMS.get(model_name, {}),
                "features": ACTIVE_FEATURES,
                "cv_splits": CFG["cv_splits"],
                "training_weights": "class_only_primary",
            },
        ),
    }
    if stage_is_valid(stage, [path], parameters):
        return normalize_probabilities(np.load(path)["proba"])
    proba = normalize_probabilities(model.predict_proba(X_split)).astype(np.float32)
    if proba.shape != (len(X_split), 4):
        raise AssertionError(f"{model_name} produced invalid probability shape {proba.shape}")
    if not np.allclose(proba.sum(axis=1), 1.0, atol=1e-5):
        raise AssertionError(f"{model_name} probabilities do not sum to one")
    atomic_savez(path, proba=proba)
    mark_stage_complete(stage, [path], parameters)
    return proba


CALIB_RAW_PROBA = {}
for name in TOP_MODEL_NAMES:
    CALIB_RAW_PROBA[name] = cached_model_probabilities(
        name, TRAINED_MODELS[name], "calibration", X_calib
    )


In [ ]:
# 9B. Fit/load temperature scalers and choose ensemble weight on calibration set
calibration_file = DIRS["models"] / "calibration_and_ensemble.json"
calibration_params = {
    "top_models": TOP_MODEL_NAMES,
    "calibrator": "single_temperature_weighted_logloss",
    "ensemble_grid": 101,
    "probability_policy": PROBABILITY_POLICY_VERSION,
}

if stage_is_valid("calibration", [calibration_file], calibration_params):
    calibration_config = json.loads(calibration_file.read_text())
    print("Loaded compatible calibration checkpoint.")
else:
    temperatures = {}
    calibrated = {}
    calibration_rows = []
    for name in TOP_MODEL_NAMES:
        raw_proba = CALIB_RAW_PROBA[name]
        scaler_t = TemperatureScaler().fit(
            raw_proba, y_calib, sample_weight=w_calib
        )
        cal_proba = scaler_t.transform(raw_proba)
        temperatures[name] = scaler_t.temperature
        calibrated[name] = cal_proba
        raw_metrics = metric_bundle(y_calib, raw_proba, w_calib, prefix="raw_")
        cal_metrics = metric_bundle(y_calib, cal_proba, w_calib, prefix="cal_")
        calibration_rows.append({
            "model": name,
            "temperature": scaler_t.temperature,
            **raw_metrics,
            **cal_metrics,
        })

    top1 = TOP_MODEL_NAMES[0]
    if len(TOP_MODEL_NAMES) >= 2:
        top2 = TOP_MODEL_NAMES[1]
        best_alpha, best_loss = None, np.inf
        for alpha in np.linspace(0, 1, 101):
            mix = alpha * calibrated[top1] + (1 - alpha) * calibrated[top2]
            loss = log_loss(
                y_calib, mix, labels=CLASS_ORDER,
                sample_weight=normalized_weights(w_calib)
            )
            if loss < best_loss:
                best_alpha, best_loss = float(alpha), float(loss)
        top1_loss = log_loss(
            y_calib, calibrated[top1], labels=CLASS_ORDER,
            sample_weight=normalized_weights(w_calib)
        )
        use_ensemble = best_loss < (top1_loss - 1e-4)
    else:
        top2 = None
        best_alpha = 1.0
        best_loss = np.nan
        top1_loss = log_loss(
            y_calib, calibrated[top1], labels=CLASS_ORDER,
            sample_weight=normalized_weights(w_calib)
        )
        use_ensemble = False

    calibration_config = {
        "top_models": TOP_MODEL_NAMES,
        "temperatures": temperatures,
        "top1": top1,
        "top2": top2,
        "ensemble_alpha_for_top1": best_alpha,
        "top1_weighted_logloss": float(top1_loss),
        "ensemble_weighted_logloss": float(best_loss) if pd.notna(best_loss) else None,
        "use_ensemble_as_primary": bool(use_ensemble),
        "primary_name": "calibrated_ensemble" if use_ensemble else f"calibrated_{top1}",
    }
    atomic_write_json(calibration_config, calibration_file)
    atomic_to_parquet(
        pd.DataFrame(calibration_rows),
        DIRS["tables"] / "calibration_set_metrics.parquet",
        index=False,
    )
    mark_stage_complete("calibration", [calibration_file], calibration_params)

display(pd.DataFrame([calibration_config]))


In [ ]:
# 9C. Serializable calibrated probability predictor
class CalibratedProbabilityPredictor:
    def __init__(self, models, temperatures, top1, top2=None, alpha=1.0, use_ensemble=False):
        self.models = models
        self.temperatures = temperatures
        self.top1 = top1
        self.top2 = top2
        self.alpha = float(alpha)
        self.use_ensemble = bool(use_ensemble)
        self.classes_ = np.array(CLASS_ORDER)

    def _calibrated(self, name, X):
        raw = self.models[name].predict_proba(X)
        return TemperatureScaler(self.temperatures[name]).transform(raw)

    def predict_proba(self, X):
        first = self._calibrated(self.top1, X)
        if self.use_ensemble and self.top2 is not None:
            second = self._calibrated(self.top2, X)
            return normalize_probabilities(
                self.alpha * first + (1 - self.alpha) * second
            )
        return normalize_probabilities(first)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


PRIMARY_PREDICTOR = CalibratedProbabilityPredictor(
    models={name: TRAINED_MODELS[name] for name in TOP_MODEL_NAMES},
    temperatures=calibration_config["temperatures"],
    top1=calibration_config["top1"],
    top2=calibration_config["top2"],
    alpha=calibration_config["ensemble_alpha_for_top1"],
    use_ensemble=calibration_config["use_ensemble_as_primary"],
)
PRIMARY_NAME = calibration_config["primary_name"]
primary_model_path = DIRS["models"] / "primary_calibrated_predictor.joblib"
primary_parameters = {"calibration_config": calibration_config}
if not stage_is_valid("primary_predictor", [primary_model_path], primary_parameters):
    atomic_joblib_dump(PRIMARY_PREDICTOR, primary_model_path)
    mark_stage_complete("primary_predictor", [primary_model_path], primary_parameters)
else:
    PRIMARY_PREDICTOR = joblib.load(primary_model_path)
print("Locked primary predictor:", PRIMARY_NAME)


## 10. Locked test evaluation

All modelling, ranking, calibration and ensemble choices are complete before this section. The test set is evaluated once and is never used to change the model.


In [ ]:
# 10A. Explicitly gated locked-test predictions and metrics
def current_stage_manifest_complete(stage):
    path = stage_manifest_path(stage)
    if not path.exists():
        return False
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return (
        payload.get("status") == "complete"
        and payload.get("pipeline_version") == PIPELINE_VERSION
        and payload.get("run_fingerprint") == RUN_FINGERPRINT
    )


TEST_GATE_STATUS = {
    "paper_mode": RUN_MODE == "paper",
    "nested_cv_manifest": current_stage_manifest_complete("nested_cv"),
    "geographic_validation_manifest": current_stage_manifest_complete(
        "geographic_validation"
    ),
}
if UNLOCK_FINAL_TEST and RUN_MODE != "paper":
    raise RuntimeError("The final test cannot be unlocked in development mode.")
missing_test_gates = [name for name, passed in TEST_GATE_STATUS.items() if not passed]
if UNLOCK_FINAL_TEST and missing_test_gates:
    raise RuntimeError(
        "Final-test unlock refused. Complete the first paper pass and rerun with "
        f"UNLOCK_FINAL_TEST=True. Missing gates: {missing_test_gates}"
    )
FINAL_TEST_UNLOCKED = bool(UNLOCK_FINAL_TEST and not missing_test_gates)

if not FINAL_TEST_UNLOCKED:
    PRIMARY_TEST_PROBA = np.empty((0, len(CLASS_ORDER)), dtype=np.float64)
    PRIMARY_TEST_PRED = np.array([], dtype=int)
    locked_test_metrics = pd.DataFrame()
    primary_row = {}
    print("Final test remains locked. Gate status:", TEST_GATE_STATUS)
else:
    test_predictions_file = DIRS["predictions"] / "locked_test_predictions.npz"
    test_stage_params = {
        "primary_name": PRIMARY_NAME,
        "calibration_config": calibration_config,
        "probability_policy": PROBABILITY_POLICY_VERSION,
        "split_random_state": SPLIT_RANDOM_STATE,
    }

    if stage_is_valid("locked_test", [test_predictions_file], test_stage_params):
        loaded = np.load(test_predictions_file)
        PRIMARY_TEST_PROBA = normalize_probabilities(loaded["primary_proba"])
        print("Loaded locked-test prediction checkpoint.")
    else:
        PRIMARY_TEST_PROBA = normalize_probabilities(
            PRIMARY_PREDICTOR.predict_proba(X_test)
        ).astype(np.float32)
        atomic_savez(test_predictions_file, primary_proba=PRIMARY_TEST_PROBA)
        mark_stage_complete("locked_test", [test_predictions_file], test_stage_params)

    assert PRIMARY_TEST_PROBA.shape == (len(X_test), 4)
    assert np.allclose(PRIMARY_TEST_PROBA.sum(axis=1), 1.0, atol=1e-5)
    PRIMARY_TEST_PRED = PRIMARY_TEST_PROBA.argmax(axis=1)

    test_rows = []
    for name, model in TRAINED_MODELS.items():
        proba = cached_model_probabilities(name, model, "test", X_test)
        row = {
            "candidate": name,
            "role": "fixed_base_model",
            **metric_bundle(y_test, proba),
            **metric_bundle(y_test, proba, w_test, prefix="survey_weighted_"),
        }
        test_rows.append(row)

    primary_row = {
        "candidate": PRIMARY_NAME,
        "role": "pre_specified_primary",
        **metric_bundle(y_test, PRIMARY_TEST_PROBA),
        **metric_bundle(y_test, PRIMARY_TEST_PROBA, w_test, prefix="survey_weighted_"),
    }
    test_rows.append(primary_row)
    locked_test_metrics = pd.DataFrame(test_rows)
    atomic_to_parquet(
        locked_test_metrics,
        DIRS["tables"] / "locked_test_metrics.parquet",
        index=False,
    )
    display(
        locked_test_metrics[[
            "candidate", "role", "macro_f1", "balanced_accuracy",
            "macro_auroc", "macro_auprc", "recall_class_3",
            "auprc_class_3", "log_loss", "brier", "ece",
            "quadratic_kappa", "ordinal_mae",
        ]].round(4)
    )
    print(classification_report(
        y_test, PRIMARY_TEST_PRED,
        labels=CLASS_ORDER,
        target_names=[CLASS_LABELS[c] for c in CLASS_ORDER],
        digits=4,
        zero_division=0,
    ))


In [ ]:
if not FINAL_TEST_UNLOCKED:
    print("Skipped: the final test remains locked.")
else:
    # 10B. Confusion matrices and per-class ROC/PR curves
    cm = confusion_matrix(y_test, PRIMARY_TEST_PRED, labels=CLASS_ORDER)
    cm_norm = confusion_matrix(
        y_test, PRIMARY_TEST_PRED, labels=CLASS_ORDER, normalize="true"
    )
    class_names = [CLASS_LABELS[c] for c in CLASS_ORDER]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names, ax=axes[0]
    )
    axes[0].set_title("Locked test confusion matrix — counts")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Observed")
    sns.heatmap(
        cm_norm, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1,
        xticklabels=class_names, yticklabels=class_names, ax=axes[1]
    )
    axes[1].set_title("Locked test confusion matrix — row-normalised")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Observed")
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "03_locked_test_confusion.png", bbox_inches="tight")
    plt.show()

    y_test_bin = label_binarize(y_test, classes=CLASS_ORDER)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for c in CLASS_ORDER:
        fpr, tpr, _ = roc_curve(y_test_bin[:, c], PRIMARY_TEST_PROBA[:, c])
        precision, recall, _ = precision_recall_curve(
            y_test_bin[:, c], PRIMARY_TEST_PROBA[:, c]
        )
        auc_c = roc_auc_score(y_test_bin[:, c], PRIMARY_TEST_PROBA[:, c])
        ap_c = average_precision_score(y_test_bin[:, c], PRIMARY_TEST_PROBA[:, c])
        axes[0].plot(fpr, tpr, color=CLASS_COLORS[c], label=f"{CLASS_LABELS[c]} AUC={auc_c:.3f}")
        axes[1].plot(recall, precision, color=CLASS_COLORS[c], label=f"{CLASS_LABELS[c]} AP={ap_c:.3f}")
    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
    axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="One-vs-rest ROC")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="One-vs-rest precision–recall")
    for ax in axes:
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "04_locked_test_roc_pr.png", bbox_inches="tight")
    plt.show()


In [ ]:
if not FINAL_TEST_UNLOCKED:
    print("Skipped: the final test remains locked.")
else:
    # 10C. Reliability diagrams after calibration
    def reliability_points(y_true, probabilities, class_code, n_bins=10, sample_weight=None):
        y_binary = (np.asarray(y_true) == class_code).astype(float)
        p = np.asarray(probabilities)[:, class_code]
        w = np.ones(len(p)) if sample_weight is None else normalized_weights(sample_weight)
        edges = np.linspace(0, 1, n_bins + 1)
        rows = []
        for b, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
            mask = (p >= lo) & (p <= hi if b == n_bins - 1 else p < hi)
            if not mask.any():
                continue
            rows.append({
                "bin": b,
                "mean_predicted": np.average(p[mask], weights=w[mask]),
                "observed": np.average(y_binary[mask], weights=w[mask]),
                "n": int(mask.sum()),
                "weighted_mass": float(w[mask].sum()),
            })
        return pd.DataFrame(rows)


    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    reliability_all = []
    for ax, c in zip(axes.flat, CLASS_ORDER):
        rel = reliability_points(
            y_test, PRIMARY_TEST_PROBA, c, n_bins=12, sample_weight=w_test
        )
        rel["class_code"] = c
        reliability_all.append(rel)
        ax.plot(rel["mean_predicted"], rel["observed"], "o-", color=CLASS_COLORS[c])
        ax.plot([0, 1], [0, 1], "k--", linewidth=1)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_title(CLASS_LABELS[c])
        ax.set_xlabel("Predicted probability")
        ax.set_ylabel(f"{DESCRIPTIVE_LABEL} observed frequency")
    fig.suptitle(f"Locked-test reliability diagrams: {PRIMARY_NAME}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "05_locked_test_calibration.png", bbox_inches="tight")
    plt.show()
    atomic_to_parquet(
        pd.concat(reliability_all, ignore_index=True),
        DIRS["tables"] / "locked_test_reliability.parquet",
        index=False,
    )


## 11. Model-comparison inference

Friedman and paired Wilcoxon tests use identical development-CV folds. They are secondary/exploratory because the best model was selected on the same development folds. Test-set results remain primarily descriptive with uncertainty intervals.


In [ ]:
# 11A. Fold-level statistical comparison
common_models = cv_summary["model"].tolist()
fold_metric_matrix = pd.DataFrame({
    name: CV_TABLES[name].sort_values("fold")["macro_f1"].to_numpy()
    for name in common_models
})
atomic_to_parquet(
    fold_metric_matrix.reset_index(names="fold"),
    DIRS["tables"] / "cv_fold_metric_matrix.parquet",
    index=False,
)

statistical_results = {}
if fold_metric_matrix.shape[0] >= 3 and fold_metric_matrix.shape[1] >= 3:
    friedman = friedmanchisquare(*[
        fold_metric_matrix[col].values for col in fold_metric_matrix.columns
    ])
    statistical_results["friedman"] = {
        "statistic": float(friedman.statistic),
        "p_value": float(friedman.pvalue),
        "metric": "macro_f1",
    }

reference = cv_summary.iloc[0]["model"]
pairwise_rows = []
for comparator in common_models:
    if comparator == reference:
        continue
    try:
        test = wilcoxon(
            fold_metric_matrix[reference],
            fold_metric_matrix[comparator],
            alternative="two-sided",
            zero_method="wilcox",
        )
        p = float(test.pvalue)
        stat = float(test.statistic)
    except ValueError:
        p, stat = 1.0, 0.0
    pairwise_rows.append({
        "reference": reference,
        "comparator": comparator,
        "statistic": stat,
        "raw_p": p,
    })

pairwise = pd.DataFrame(pairwise_rows).sort_values("raw_p").reset_index(drop=True)
if not pairwise.empty:
    m = len(pairwise)
    adjusted = np.maximum.accumulate(
        [(m - rank) * p for rank, p in enumerate(pairwise["raw_p"])]
    )
    pairwise["holm_p"] = np.minimum(adjusted, 1.0)
    atomic_to_parquet(
        pairwise, DIRS["tables"] / "cv_pairwise_wilcoxon_holm.parquet", index=False
    )
statistical_results["pairwise_reference"] = reference
atomic_write_json(statistical_results, DIRS["tables"] / "cv_statistical_tests.json")

print(json.dumps(statistical_results, indent=2))
display(pairwise.round(5) if not pairwise.empty else pairwise)


## 12. Resumable survey-design bootstrap uncertainty

The fitted primary model remains frozen. The locked-test predictions are
resampled by drawing whole PSUs **with replacement within each observed
stratum**. Replicates are written every ten iterations, so a runtime failure
resumes from the last completed replicate. Percentile intervals quantify
internal sampling uncertainty; they are not external-validation intervals.


In [ ]:
if not FINAL_TEST_UNLOCKED:
    bootstrap_df = pd.DataFrame()
    bootstrap_ci = pd.DataFrame()
    print("Skipped: the final test remains locked.")
else:
    import os
    # 12A. Resumable stratified PSU-cluster bootstrap
    bootstrap_path = DIRS["tables"] / "test_stratified_psu_bootstrap.parquet"
    bootstrap_ci_path = DIRS["tables"] / "test_stratified_psu_bootstrap_ci.parquet"
    bootstrap_params = {
        "runs": CFG["bootstrap_runs"],
        "seed": RANDOM_STATE,
        "unit": "PSU_within_observed_test_stratum",
        "primary_name": PRIMARY_NAME,
    }

    # Check if the file exists and is not empty before trying to read it
    if bootstrap_path.exists() and os.path.getsize(bootstrap_path) > 0 and "bootstrap" not in FORCE_STAGES:
        bootstrap_df = pd.read_parquet(bootstrap_path)
    else:
        bootstrap_df = pd.DataFrame()

    completed_bootstrap = set(
        bootstrap_df["replicate"].astype(int).tolist()
        if not bootstrap_df.empty else []
    )
    test_design = analysis_df.iloc[TEST_POS][["psu", "strata"]].reset_index(drop=True)
    psu_to_pos = {
        psu: np.flatnonzero(test_design["psu"].to_numpy() == psu)
        for psu in pd.unique(test_design["psu"].dropna())
    }
    psu_strata = (
        test_design.dropna(subset=["psu", "strata"])
        .drop_duplicates("psu").groupby("strata", observed=True)["psu"].apply(list)
    )
    if psu_strata.empty:
        psu_strata = pd.Series({"all_test": list(psu_to_pos)})

    new_rows = []
    for b in tqdm(range(CFG["bootstrap_runs"]), desc="Stratified PSU bootstrap"):
        if b in completed_bootstrap:
            continue
        rng = np.random.default_rng(RANDOM_STATE + 10_000 + b)
        sampled_positions = []
        sampled_psu_count = 0
        for _, psus in psu_strata.items():
            psus = np.asarray(psus, dtype=object)
            sampled = rng.choice(psus, size=len(psus), replace=True)
            sampled_psu_count += len(sampled)
            sampled_positions.extend(psu_to_pos[p] for p in sampled)
        sampled_pos = np.concatenate(sampled_positions)
        metrics_b = metric_bundle(
            y_test.iloc[sampled_pos],
            PRIMARY_TEST_PROBA[sampled_pos],
            w_test.iloc[sampled_pos],
        )
        metrics_b.update({
            "replicate": b,
            "n_rows_with_multiplicity": len(sampled_pos),
            "n_psu_draws": sampled_psu_count,
            "n_strata": len(psu_strata),
        })
        new_rows.append(metrics_b)

        if len(new_rows) >= 10:
            bootstrap_df = pd.concat(
                [bootstrap_df, pd.DataFrame(new_rows)], ignore_index=True
            ).drop_duplicates("replicate", keep="last").sort_values("replicate")
            atomic_to_parquet(bootstrap_df, bootstrap_path, index=False)
            new_rows = []

    if new_rows:
        bootstrap_df = pd.concat(
            [bootstrap_df, pd.DataFrame(new_rows)], ignore_index=True
        ).drop_duplicates("replicate", keep="last").sort_values("replicate")
        atomic_to_parquet(bootstrap_df, bootstrap_path, index=False)

    if len(bootstrap_df) < CFG["bootstrap_runs"]:
        raise RuntimeError(
            f"Bootstrap incomplete: {len(bootstrap_df)}/{CFG['bootstrap_runs']} replicates"
        )

    nonmetric_columns = {
        "replicate", "n_rows_with_multiplicity", "n_psu_draws", "n_strata"
    }
    metric_columns = [c for c in bootstrap_df.columns if c not in nonmetric_columns]
    bootstrap_ci = pd.DataFrame([
        {
            "metric": metric,
            "estimate": primary_row.get(f"survey_weighted_{metric}", np.nan),
            "bootstrap_median": bootstrap_df[metric].median(),
            "ci_2.5": bootstrap_df[metric].quantile(0.025),
            "ci_97.5": bootstrap_df[metric].quantile(0.975),
        }
        for metric in metric_columns
    ])
    atomic_to_parquet(bootstrap_ci, bootstrap_ci_path, index=False)
    mark_stage_complete(
        "bootstrap", [bootstrap_path, bootstrap_ci_path], bootstrap_params,
        extra={"completed_replicates": len(bootstrap_df), "n_strata": len(psu_strata)},
    )

    display(
        bootstrap_ci.loc[
            bootstrap_ci["metric"].isin([
                "macro_f1", "balanced_accuracy", "macro_auroc", "macro_auprc",
                "recall_class_3", "auprc_class_3", "brier", "ece",
                "quadratic_kappa", "ordinal_mae",
            ])
        ].round(4)
    )


In [ ]:
if not FINAL_TEST_UNLOCKED:
    ci_plot = pd.DataFrame()
    print("Skipped: the final test remains locked.")
else:
    ci_plot = bootstrap_ci.loc[
        bootstrap_ci["metric"].isin([
            "macro_f1", "balanced_accuracy", "macro_auprc",
            "recall_class_3", "auprc_class_3", "quadratic_kappa",
        ])
    ].copy()
    ci_plot["label"] = ci_plot["metric"].replace({
        "macro_f1": "Macro F1",
        "balanced_accuracy": "Balanced accuracy",
        "macro_auprc": "Macro AUPRC",
        "recall_class_3": "Severe recall",
        "auprc_class_3": "Severe AUPRC",
        "quadratic_kappa": "Quadratic kappa",
    })
    fig, ax = plt.subplots(figsize=(9, 5))
    ypos = np.arange(len(ci_plot))
    ax.errorbar(
        ci_plot["estimate"], ypos,
        xerr=[
            (ci_plot["estimate"] - ci_plot["ci_2.5"]).clip(lower=0),
            (ci_plot["ci_97.5"] - ci_plot["estimate"]).clip(lower=0),
        ],
        fmt="o", capsize=4, color="#264653"
    )
    ax.set_yticks(ypos)
    ax.set_yticklabels(ci_plot["label"])
    ax.set_xlabel("Estimate with 95% PSU-bootstrap interval")
    ax.set_title("Locked-test performance uncertainty")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "06_bootstrap_intervals.png", bbox_inches="tight")
    plt.show()

## 13. Stakeholder-defined cost-sensitive decision analysis

These matrices are not WHO-prescribed. They are transparent hypothetical loss scenarios. Any clinical deployment would require stakeholder elicitation, external validation and prospective impact evaluation.

For each calibrated probability vector, the chosen action minimises expected loss:

\[
\hat a(x)=\arg\min_a \sum_j C_{j,a}P(Y=j\mid X=x).
\]


In [ ]:
if not FINAL_TEST_UNLOCKED:
    cost_analysis = pd.DataFrame()
    print("Skipped: the final test remains locked.")
else:
    # 13A. Expected-cost decision policies
    COST_SCENARIOS = {
        "symmetric_ordinal": np.array([
            [0, 1, 2, 3],
            [1, 0, 1, 2],
            [2, 1, 0, 1],
            [3, 2, 1, 0],
        ], dtype=float),
        "moderate_severe_priority": np.array([
            [0, 1, 2, 3],
            [1, 0, 1, 2],
            [5, 2, 0, 1],
            [12, 8, 3, 0],
        ], dtype=float),
        "high_severe_sensitivity": np.array([
            [0, 1, 3, 5],
            [1, 0, 2, 4],
            [7, 3, 0, 2],
            [20, 12, 5, 0],
        ], dtype=float),
    }


    def expected_cost_predictions(proba, cost_matrix):
        # C[true_class, predicted_action]
        expected_cost = np.asarray(proba) @ np.asarray(cost_matrix)
        return expected_cost.argmin(axis=1)


    def observed_average_cost(y_true, pred, cost_matrix, sample_weight=None):
        costs = np.asarray(cost_matrix)[
            np.asarray(y_true, dtype=int), np.asarray(pred, dtype=int)
        ]
        return float(np.average(costs, weights=sample_weight))


    cost_rows = []
    argmax_pred = PRIMARY_TEST_PROBA.argmax(axis=1)
    for scenario, matrix in COST_SCENARIOS.items():
        policy_pred = expected_cost_predictions(PRIMARY_TEST_PROBA, matrix)
        for policy_name, pred in [
            ("argmax", argmax_pred),
            ("minimum_expected_cost", policy_pred),
        ]:
            row = {
                "scenario": scenario,
                "policy": policy_name,
                "survey_weighted_average_cost": observed_average_cost(
                    y_test, pred, matrix, normalized_weights(w_test)
                ),
                "macro_f1": f1_score(
                    y_test, pred, average="macro",
                    sample_weight=normalized_weights(w_test), zero_division=0
                ),
                "severe_recall": recall_score(
                    y_test, pred, labels=[3], average="macro",
                    sample_weight=normalized_weights(w_test), zero_division=0
                ),
                "moderate_or_severe_recall": recall_score(
                    (np.asarray(y_test) >= 2).astype(int),
                    (np.asarray(pred) >= 2).astype(int),
                    sample_weight=normalized_weights(w_test),
                    zero_division=0,
                ),
            }
            cost_rows.append(row)

    cost_analysis = pd.DataFrame(cost_rows)
    atomic_to_parquet(
        cost_analysis, DIRS["tables"] / "cost_sensitivity.parquet", index=False
    )
    display(cost_analysis.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (scenario, matrix) in zip(axes, COST_SCENARIOS.items()):
        sns.heatmap(
            matrix, annot=True, fmt=".0f", cmap="Reds",
            xticklabels=[CLASS_LABELS[c] for c in CLASS_ORDER],
            yticklabels=[CLASS_LABELS[c] for c in CLASS_ORDER],
            ax=ax, cbar=False,
        )
        ax.set_title(scenario.replace("_", " ").title())
        ax.set_xlabel("Predicted action")
        ax.set_ylabel("Observed class")
    plt.tight_layout()
    plt.savefig(DIRS["figures"] / "07_cost_scenarios.png", bbox_inches="tight")
    plt.show()


## 14. Subgroup robustness and equity diagnostics

Locked-test performance is reported only for groups meeting pre-specified
minimum sample and severe-case counts. Diagnostics now include breastfeeding,
amenorrhea, pregnancy-termination history, health insurance, education and
diet diversity in addition to residence, caste, religion, region and state.
Differences are descriptive model-performance gaps, not evidence of biological
group effects.


In [ ]:
if not FINAL_TEST_UNLOCKED:
    subgroup_results = pd.DataFrame()
    disparity = pd.DataFrame()
    print("Skipped: the final test remains locked.")
else:
    test_context = analysis_df.iloc[TEST_POS].reset_index(drop=True).copy()

    def subgroup_metric_table(context, group_col, y_true, proba, weights_local):
        rows = []
        # Fill NaN values in the grouping column with a placeholder to avoid
        # ValueError: Categorical categories cannot be null, which can occur
        # if pandas internally attempts to create a Categorical index with NaN as a category.
        grouping_series = context[group_col].fillna("NaN_Placeholder")

        for value, idx in context.groupby(grouping_series, dropna=False).groups.items():
            pos = np.asarray(list(idx), dtype=int)
            if len(pos) < MIN_SUBGROUP_N:
                continue
            severe_n = int((np.asarray(y_true)[pos] == 3).sum())
            if severe_n < MIN_SUBGROUP_SEVERE:
                continue
            metrics_local = metric_bundle(
                np.asarray(y_true)[pos],
                np.asarray(proba)[pos],
                np.asarray(weights_local)[pos],
            )
            rows.append({
                "group_variable": group_col,
                "group_value": str(value),
                "n": len(pos),
                "severe_n": severe_n,
                **metrics_local,
            })
        return pd.DataFrame(rows)


    subgroup_tables = []
    for group_col in [
        "currently_pregnant", "currently_breastfeeding",
        "currently_amenorrheic", "pregnancy_termination_history",
        "health_insurance", "education_group", "diet_diversity_group",
        "residence", "caste", "religion", "region_group", "state_code",
    ]:
        # The fillna within subgroup_metric_table handles NaNs, so we only need to check if the column exists.
        if group_col in test_context:
            table = subgroup_metric_table(
                test_context, group_col, y_test, PRIMARY_TEST_PROBA, w_test
            )
            if not table.empty:
                subgroup_tables.append(table)

    subgroup_results = (
        pd.concat(subgroup_tables, ignore_index=True)
        if subgroup_tables else pd.DataFrame()
    )
    if not subgroup_results.empty:
        atomic_to_parquet(
            subgroup_results, DIRS["tables"] / "subgroup_performance.parquet", index=False
        )
        disparity = (
            subgroup_results.groupby("group_variable")
            .agg(
                groups=("group_value", "nunique"),
                macro_f1_min=("macro_f1", "min"),
                macro_f1_max=("macro_f1", "max"),
                severe_recall_min=("recall_class_3", "min"),
                severe_recall_max=("recall_class_3", "max"),
                ece_max=("ece", "max"),
            )
            .reset_index()
        )
        disparity["macro_f1_gap"] = disparity["macro_f1_max"] - disparity["macro_f1_min"]
        disparity["severe_recall_gap"] = (
            disparity["severe_recall_max"] - disparity["severe_recall_min"]
        )
        atomic_to_parquet(
            disparity, DIRS["tables"] / "subgroup_disparity_summary.parquet", index=False
        )
        display(disparity.round(4))
    else:
        print("No subgroup met the pre-specified support thresholds.")

In [ ]:
# 14B. Observed survey-weighted state prevalence (not predicted prevalence)
STATE_UT_NAMES = {
    1: "Jammu & Kashmir", 2: "Himachal Pradesh", 3: "Punjab",
    4: "Chandigarh", 5: "Uttarakhand", 6: "Haryana", 7: "Delhi",
    8: "Rajasthan", 9: "Uttar Pradesh", 10: "Bihar", 11: "Sikkim",
    12: "Arunachal Pradesh", 13: "Nagaland", 14: "Manipur",
    15: "Mizoram", 16: "Tripura", 17: "Meghalaya", 18: "Assam",
    19: "West Bengal", 20: "Jharkhand", 21: "Odisha",
    22: "Chhattisgarh", 23: "Madhya Pradesh", 24: "Gujarat",
    25: "Daman & Diu", 26: "Dadra & Nagar Haveli", 27: "Maharashtra",
    28: "Andhra Pradesh", 29: "Karnataka", 30: "Goa",
    31: "Lakshadweep", 32: "Kerala", 33: "Tamil Nadu",
    34: "Puducherry", 35: "Andaman & Nicobar Islands",
    36: "Telangana", 37: "Ladakh",
}


def state_label(code):
    if pd.isna(code):
        return "Missing"
    integer_code = int(code)
    source_mapping = SOURCE_VALUE_LABELS.get("v024", {})
    for key in [str(integer_code), str(float(integer_code))]:
        if key in source_mapping:
            return source_mapping[key]
    return STATE_UT_NAMES.get(integer_code, f"State/UT code {integer_code}")


state_rows = []
for state, subset in analysis_df.dropna(subset=["state_code"]).groupby("state_code"):
    y_state = subset["anaemia_level"].to_numpy()
    w_state = normalized_weights(subset["sample_weight"])
    prevalence = weighted_class_prevalence(y_state, w_state)
    state_rows.append({
        "state_code": state,
        "state_name": state_label(state),
        "unweighted_n": len(subset),
        "n_psu": subset["psu"].nunique(),
        "weighted_no_anaemia_pct": 100 * prevalence[0],
        "weighted_mild_pct": 100 * prevalence[1],
        "weighted_moderate_pct": 100 * prevalence[2],
        "weighted_severe_pct": 100 * prevalence[3],
        "weighted_moderate_or_severe_pct": 100 * (prevalence[2] + prevalence[3]),
    })
state_prevalence = pd.DataFrame(state_rows).sort_values(
    "weighted_moderate_or_severe_pct", ascending=False
)
atomic_to_parquet(
    state_prevalence, DIRS["tables"] / "observed_state_prevalence.parquet", index=False
)
display(state_prevalence.head(15).round(3))

top_states = state_prevalence.head(20).sort_values(
    "weighted_moderate_or_severe_pct"
)
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    top_states["state_name"],
    top_states["weighted_moderate_or_severe_pct"],
    color="#D95F02",
)
ax.set_xlabel(f"{DESCRIPTIVE_LABEL} moderate or severe anaemia (%)")
ax.set_title(f"Observed NFHS-5 state prevalence — top 20 ({DESCRIPTIVE_LABEL})")
plt.tight_layout()
plt.savefig(DIRS["figures"] / "08_observed_state_prevalence.png", bbox_inches="tight")
plt.show()


## 15. Haemoglobin-derived outcome coding audit


In [ ]:
# 15A. Outcome-coding reconstruction with pregnancy-duration limitation
def who_severity_from_adjusted_hb(hb, pregnant, pregnancy_month=None):
    hb = np.asarray(hb, dtype=float)
    pregnant_num = pd.to_numeric(pd.Series(pregnant), errors="coerce").to_numpy()
    month = (
        np.full(len(hb), np.nan)
        if pregnancy_month is None else np.asarray(pregnancy_month, dtype=float)
    )
    out = np.full(len(hb), np.nan)

    nonpreg = (pregnant_num == 0) & np.isfinite(hb)
    out[nonpreg & (hb >= 12.0)] = 0
    out[nonpreg & (hb >= 11.0) & (hb < 12.0)] = 1
    out[nonpreg & (hb >= 8.0) & (hb < 11.0)] = 2
    out[nonpreg & (hb < 8.0)] = 3

    preg_known = (pregnant_num == 1) & np.isfinite(hb) & np.isfinite(month)
    trimester2 = preg_known & month.between(4, 6) if isinstance(month, pd.Series) else (
        preg_known & (month >= 4) & (month <= 6)
    )
    trimester13 = preg_known & ~trimester2
    out[trimester13 & (hb >= 11.0)] = 0
    out[trimester13 & (hb >= 10.0) & (hb < 11.0)] = 1
    out[trimester13 & (hb >= 7.0) & (hb < 10.0)] = 2
    out[trimester13 & (hb < 7.0)] = 3
    out[trimester2 & (hb >= 10.5)] = 0
    out[trimester2 & (hb >= 9.5) & (hb < 10.5)] = 1
    out[trimester2 & (hb >= 7.0) & (hb < 9.5)] = 2
    out[trimester2 & (hb < 7.0)] = 3
    return out


if analysis_df["adjusted_hb_gdl"].notna().any():
    pregnancy_month = analysis_df.get(
        "pregnancy_month", pd.Series(np.nan, index=analysis_df.index)
    )
    who_target = who_severity_from_adjusted_hb(
        analysis_df["adjusted_hb_gdl"],
        analysis_df["currently_pregnant"],
        pregnancy_month,
    )
    mask = np.isfinite(who_target)
    observed = analysis_df.loc[mask, "anaemia_level"].astype(int).to_numpy()
    reconstructed = who_target[mask].astype(int)
    sensitivity_weights = normalized_weights(analysis_df.loc[mask, "sample_weight"])
    weighted_matrix = confusion_matrix(
        observed, reconstructed, labels=CLASS_ORDER,
        sample_weight=sensitivity_weights, normalize="all",
    )
    sensitivity_table = pd.DataFrame(
        weighted_matrix,
        index=[f"NFHS_{CLASS_LABELS[c]}" for c in CLASS_ORDER],
        columns=[f"Threshold_{CLASS_LABELS[c]}" for c in CLASS_ORDER],
    )
    pregnant_with_hb = (
        numeric(analysis_df["currently_pregnant"]).eq(1)
        & analysis_df["adjusted_hb_gdl"].notna()
    )
    sensitivity_summary = {
        "eligible_n": int(mask.sum()),
        "nonpregnant_eligible_n": int((
            numeric(analysis_df["currently_pregnant"]).eq(0)
            & analysis_df["adjusted_hb_gdl"].notna()
        ).sum()),
        "pregnant_excluded_missing_duration_n": int(pregnant_with_hb.sum()),
        "weighted_quadratic_kappa": float(cohen_kappa_score(
            observed, reconstructed, weights="quadratic",
            sample_weight=sensitivity_weights,
        )),
        "weighted_exact_agreement": float(np.average(
            observed == reconstructed, weights=sensitivity_weights
        )),
        "interpretation": (
            "Expected reconstruction agreement because v457 is derived from "
            "adjusted haemoglobin; this is a coding audit, not independent validation."
        ),
        "limitation": (
            "Uses DHS-adjusted haemoglobin. The final extract lacks v214, so "
            "pregnant women cannot be assigned trimester-specific thresholds "
            "and are excluded from this reconstruction."
        ),
    }
    atomic_to_parquet(
        sensitivity_table.reset_index(names="nfhs_class"),
        DIRS["tables"] / "outcome_coding_audit_weighted_crosstab.parquet",
        index=False,
    )
    atomic_write_json(
        sensitivity_summary,
        DIRS["tables"] / "outcome_coding_audit_summary.json",
    )
    display(sensitivity_table.round(4))
    print(json.dumps(sensitivity_summary, indent=2))
else:
    print("Outcome-coding audit skipped because valid v456 values are unavailable.")


## 16. Imbalance-strategy sensitivity

The primary pipeline uses model-native class weighting. Synthetic SMOTE after one-hot encoding would generate fractional category combinations, so it is not used as the primary method. A development-only sensitivity compares no balancing, class weighting and random oversampling using the same PSU-grouped folds.


In [ ]:
# 16A. Resumable imbalance ablation on the HPO subset
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


def make_unbalanced_estimator(model_name, params):
    estimator = make_estimator(model_name, params)
    if model_name == "random_forest":
        estimator.set_params(class_weight=None)
    elif model_name == "lightgbm":
        estimator.set_params(class_weight=None)
    elif model_name == "catboost":
        estimator.set_params(auto_class_weights=None)
    return estimator


def make_imbalance_pipeline(model_name, params, strategy):
    estimator = (
        make_estimator(model_name, params)
        if strategy == "class_weight"
        else make_unbalanced_estimator(model_name, params)
    )
    steps = [("preprocess", make_preprocessor())]
    if strategy == "random_oversampling":
        steps.append(("resample", RandomOverSampler(random_state=RANDOM_STATE)))
    steps.append(("model", estimator))
    return ImbPipeline(steps)


tree_rank = [
    name for name in cv_summary["model"].tolist()
    if name in {"random_forest", "lightgbm", "xgboost", "catboost"}
]
IMBALANCE_MODEL = tree_rank[0] if tree_rank else TOP_MODEL_NAMES[0]
imbalance_path = DIRS["tables"] / "imbalance_sensitivity.parquet"
imbalance_params = {
    "model": IMBALANCE_MODEL,
    "best_params": BEST_PARAMS.get(IMBALANCE_MODEL, {}),
    "rows": len(X_hpo),
    "strategies": ["none", "class_weight", "random_oversampling"],
}

if stage_is_valid("imbalance_sensitivity", [imbalance_path], imbalance_params):
    imbalance_results = pd.read_parquet(imbalance_path)
else:
    splitter = StratifiedGroupKFold(
        n_splits=min(3, CFG["cv_splits"]),
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    rows = []
    for strategy in ["none", "class_weight", "random_oversampling"]:
        for fold, (tr, va) in enumerate(splitter.split(X_hpo, y_hpo, g_hpo)):
            pipeline = make_imbalance_pipeline(
                IMBALANCE_MODEL, BEST_PARAMS.get(IMBALANCE_MODEL, {}), strategy
            )
            fit_kwargs = {}
            if IMBALANCE_MODEL == "xgboost" and strategy == "class_weight":
                fit_kwargs["model__sample_weight"] = compute_sample_weight(
                    class_weight="balanced", y=y_hpo.iloc[tr]
                )
            pipeline.fit(X_hpo.iloc[tr], y_hpo.iloc[tr], **fit_kwargs)
            proba = pipeline.predict_proba(X_hpo.iloc[va])
            rows.append({
                "model": IMBALANCE_MODEL,
                "strategy": strategy,
                "fold": fold,
                **metric_bundle(y_hpo.iloc[va], proba),
            })
            del pipeline
    imbalance_results = pd.DataFrame(rows)
    atomic_to_parquet(imbalance_results, imbalance_path, index=False)
    mark_stage_complete(
        "imbalance_sensitivity", [imbalance_path], imbalance_params
    )

display(
    imbalance_results.groupby("strategy")[
        ["macro_f1", "recall_class_3", "auprc_class_3", "log_loss", "brier"]
    ].agg(["mean", "std"]).round(4)
)


## 17. Confirmatory nested grouped cross-validation

In paper mode, the highest-ranked tree algorithm undergoes nested PSU-grouped validation. Every outer fold has its own persistent Optuna study and completion manifest. The locked test set remains untouched.

Nested-CV metrics are uncalibrated model-development metrics; calibration is assessed separately on the protected calibration/test workflow.


In [ ]:
# 17A. Checkpointed nested grouped CV for the selected tree algorithm
NESTED_MODEL = tree_rank[0] if tree_rank else TOP_MODEL_NAMES[0]


def nested_optimize(
    model_name, X_outer_train, y_outer_train, g_outer_train,
    outer_fold, n_trials, inner_splits
):
    sample_pos = group_sample_positions(
        g_outer_train, CFG["hpo_rows"], RANDOM_STATE + outer_fold
    )
    X_local = X_outer_train.iloc[sample_pos]
    y_local = y_outer_train.iloc[sample_pos]
    g_local = g_outer_train.iloc[sample_pos]
    study_name = (
        f"{PIPELINE_VERSION}_{RUN_FINGERPRINT}_{model_name}"
        f"_nested_outer_{outer_fold}"
    )
    study = optuna.create_study(
        study_name=study_name,
        storage=OPTUNA_DB,
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE + outer_fold),
        load_if_exists=True,
    )
    completed = sum(
        t.state == optuna.trial.TrialState.COMPLETE for t in study.trials
    )

    def objective(trial):
        params = suggest_params(trial, model_name)
        score, _ = grouped_cv_score(
            model_name, params, X_local, y_local, g_local,
            pd.Series(np.ones(len(y_local)), index=y_local.index),
            n_splits=inner_splits,
        )
        return score

    remaining = max(0, n_trials - completed)
    if remaining:
        study.optimize(
            objective,
            n_trials=remaining,
            gc_after_trial=True,
            show_progress_bar=True,
            catch=(MemoryError,),
        )
    return study.best_params, float(study.best_value)


nested_summary_path = DIRS["tables"] / "nested_grouped_cv.parquet"
nested_config = {
    "enabled": CFG["nested_cv"],
    "model": NESTED_MODEL,
    "outer_splits": CFG["nested_outer"],
    "inner_splits": CFG["nested_inner"],
    "trials_per_outer_fold": CFG["nested_trials"],
}

if CFG["nested_cv"]:
    dev_pos_all = np.concatenate([TRAIN_POS, CALIB_POS])
    X_dev = X.iloc[dev_pos_all].reset_index(drop=True)
    y_dev = y.iloc[dev_pos_all].reset_index(drop=True)
    g_dev = groups.iloc[dev_pos_all].reset_index(drop=True)
    w_dev = weights.iloc[dev_pos_all].reset_index(drop=True)
    outer = StratifiedGroupKFold(
        n_splits=CFG["nested_outer"],
        shuffle=True,
        random_state=RANDOM_STATE + 77,
    )
    nested_rows = []
    for fold, (tr, va) in enumerate(outer.split(X_dev, y_dev, g_dev)):
        fold_json = DIRS["tables"] / f"nested_outer_fold_{fold}.json"
        fold_stage = f"nested_outer_{fold}"
        fold_params = {**nested_config, "fold": fold}
        if stage_is_valid(fold_stage, [fold_json], fold_params):
            nested_rows.append(json.loads(fold_json.read_text()))
            print(f"Nested outer fold {fold}: loaded")
            continue

        best_nested_params, best_inner_score = nested_optimize(
            NESTED_MODEL,
            X_dev.iloc[tr], y_dev.iloc[tr], g_dev.iloc[tr],
            outer_fold=fold,
            n_trials=CFG["nested_trials"],
            inner_splits=CFG["nested_inner"],
        )
        nested_model = make_pipeline(NESTED_MODEL, best_nested_params)
        fit_kwargs = fit_kwargs_for_model(
            NESTED_MODEL, y_dev.iloc[tr], None
        )
        t0 = time.time()
        nested_model.fit(X_dev.iloc[tr], y_dev.iloc[tr], **fit_kwargs)
        proba = nested_model.predict_proba(X_dev.iloc[va])
        row = {
            "fold": fold,
            "model": NESTED_MODEL,
            "inner_best_macro_f1": best_inner_score,
            "best_params": best_nested_params,
            "n_train": len(tr),
            "n_valid": len(va),
            "seconds": time.time() - t0,
            **metric_bundle(y_dev.iloc[va], proba),
            **metric_bundle(
                y_dev.iloc[va], proba, w_dev.iloc[va],
                prefix="survey_weighted_"
            ),
        }
        atomic_write_json(row, fold_json)
        mark_stage_complete(fold_stage, [fold_json], fold_params)
        nested_rows.append(row)
        del nested_model

    nested_results = pd.DataFrame(nested_rows).sort_values("fold")
    atomic_to_parquet(nested_results, nested_summary_path, index=False)
    mark_stage_complete(
        "nested_cv", [nested_summary_path], nested_config,
        extra={"completed_folds": len(nested_results)}
    )
    display(
        nested_results[[
            "fold", "macro_f1", "macro_auroc", "macro_auprc",
            "recall_class_3", "auprc_class_3", "quadratic_kappa"
        ]].round(4)
    )
    print("Nested-CV mean ± SD macro F1:",
          f"{nested_results['macro_f1'].mean():.4f} ± "
          f"{nested_results['macro_f1'].std(ddof=1):.4f}")
else:
    nested_results = pd.DataFrame()
    print("Nested grouped CV disabled in development mode.")


## 18. Geographic internal–external validation

The portable feature set excludes state, caste, religion and India-specific region labels. Whole states are held out via GroupKFold. This tests geographic transportability more strongly than a random split, although it is still internal–external validation within NFHS-5.


In [ ]:
# 18A. State-held-out validation using the portable feature set
PORTABLE_FEATURES = [
    c for c in FEATURE_SETS["portable"]
    if c in analysis_df and analysis_df[c].notna().any()
]
PORTABLE_NUMERIC = [c for c in PORTABLE_FEATURES if c in NUMERIC_BASE]
PORTABLE_CATEGORICAL = [c for c in PORTABLE_FEATURES if c not in PORTABLE_NUMERIC]
assert_no_target_leakage(PORTABLE_FEATURES)


def make_preprocessor_for(numeric_cols, categorical_cols):
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler(with_mean=False)),
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent", keep_empty_features=True
                )),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore", min_frequency=20, sparse_output=True
                )),
            ]), categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )


geographic_path = DIRS["tables"] / "state_heldout_validation.parquet"
geographic_config = {
    "enabled": CFG["geographic_cv"],
    "model": NESTED_MODEL,
    "features": PORTABLE_FEATURES,
    "n_splits": 5,
}

if CFG["geographic_cv"]:
    geo_frame = analysis_df.dropna(subset=["state_code"]).reset_index(drop=True)
    X_geo = geo_frame[PORTABLE_FEATURES]
    y_geo = geo_frame["anaemia_level"].astype(int)
    state_geo = geo_frame["state_code"]
    w_geo = geo_frame["sample_weight"]
    geo_splitter = GroupKFold(n_splits=5)
    geo_rows = []
    for fold, (tr, va) in enumerate(geo_splitter.split(X_geo, y_geo, state_geo)):
        fold_json = DIRS["tables"] / f"state_heldout_fold_{fold}.json"
        fold_stage = f"state_heldout_{fold}"
        fold_params = {**geographic_config, "fold": fold}
        if stage_is_valid(fold_stage, [fold_json], fold_params):
            geo_rows.append(json.loads(fold_json.read_text()))
            print(f"State-held-out fold {fold}: loaded")
            continue

        estimator = make_estimator(
            NESTED_MODEL, BEST_PARAMS.get(NESTED_MODEL, {})
        )
        pipeline = Pipeline([
            ("preprocess", make_preprocessor_for(
                PORTABLE_NUMERIC, PORTABLE_CATEGORICAL
            )),
            ("model", estimator),
        ])
        fit_kwargs = fit_kwargs_for_model(NESTED_MODEL, y_geo.iloc[tr], None)
        t0 = time.time()
        pipeline.fit(X_geo.iloc[tr], y_geo.iloc[tr], **fit_kwargs)
        proba = pipeline.predict_proba(X_geo.iloc[va])
        heldout_states = sorted(pd.unique(state_geo.iloc[va]).tolist())
        row = {
            "fold": fold,
            "heldout_state_codes": heldout_states,
            "n_train": len(tr),
            "n_valid": len(va),
            "seconds": time.time() - t0,
            **metric_bundle(y_geo.iloc[va], proba),
            **metric_bundle(
                y_geo.iloc[va], proba, w_geo.iloc[va],
                prefix="survey_weighted_"
            ),
        }
        atomic_write_json(row, fold_json)
        mark_stage_complete(fold_stage, [fold_json], fold_params)
        geo_rows.append(row)
        del pipeline

    geographic_results = pd.DataFrame(geo_rows).sort_values("fold")
    atomic_to_parquet(geographic_results, geographic_path, index=False)
    mark_stage_complete(
        "geographic_validation", [geographic_path], geographic_config
    )
    display(
        geographic_results[[
            "fold", "heldout_state_codes", "macro_f1", "macro_auprc",
            "recall_class_3", "auprc_class_3", "ece"
        ]].round(4)
    )
else:
    geographic_results = pd.DataFrame()
    print("State-held-out validation disabled in development mode.")


## 19. Held-out class-specific SHAP analysis

SHAP is calculated only for the highest-ranked completed tree model and only on
a stratified sample from the untouched test set. One-hot contributions are
aggregated back to source features for class-specific policy interpretation.
Both absolute and signed mean contributions are exported; signed values are
descriptive model effects and must not be interpreted causally.


In [ ]:
if not FINAL_TEST_UNLOCKED:
    shap_source_importance = pd.DataFrame()
    print("Skipped: the final test remains locked.")
else:
    # 19A. Robust, checkpointed SHAP computation
    SHAP_MODEL_NAME = tree_rank[0] if tree_rank else None


    def stratified_sample_positions(y_local, total_n, random_state=42):
        y_array = np.asarray(y_local)
        rng = np.random.default_rng(random_state)
        selected = []
        per_class = max(1, total_n // len(CLASS_ORDER))
        for c in CLASS_ORDER:
            available_pos = np.flatnonzero(y_array == c)
            take = min(per_class, len(available_pos))
            if take:
                selected.extend(rng.choice(available_pos, size=take, replace=False).tolist())
        selected = list(dict.fromkeys(selected))
        if len(selected) < min(total_n, len(y_array)):
            remaining = np.setdiff1d(np.arange(len(y_array)), np.asarray(selected))
            fill = min(total_n - len(selected), len(remaining))
            if fill:
                selected.extend(rng.choice(remaining, size=fill, replace=False).tolist())
        return np.asarray(selected, dtype=int)


    def normalize_multiclass_shap(values, n_samples, n_features, n_classes=4):
        if isinstance(values, list):
            arr = np.stack([np.asarray(v) for v in values], axis=-1)
        else:
            arr = np.asarray(values)
        if arr.ndim == 2:
            arr = arr[:, :, None]
        if arr.shape == (n_samples, n_features, n_classes):
            return arr
        if arr.shape == (n_samples, n_classes, n_features):
            return np.transpose(arr, (0, 2, 1))
        if arr.shape == (n_classes, n_samples, n_features):
            return np.transpose(arr, (1, 2, 0))
        raise ValueError(f"Unrecognised SHAP shape: {arr.shape}")


    def transformed_to_raw_feature(name):
        name = str(name)
        if name.startswith("num__"):
            return name.split("num__", 1)[1]
        if name.startswith("num__missingindicator_"):
            return name.split("num__missingindicator_", 1)[1] + "__missing"
        if name.startswith("cat__"):
            body = name.split("cat__", 1)[1]
            for feature in sorted(CATEGORICAL_FEATURES, key=len, reverse=True):
                if body == feature or body.startswith(feature + "_"):
                    return feature
            return body
        return name


    if SHAP_MODEL_NAME:
        shap_path = DIRS["cache"] / f"shap_{SHAP_MODEL_NAME}.npz"
        shap_importance_path = DIRS["tables"] / "shap_source_feature_importance.parquet"
        shap_params = {
            "model": SHAP_MODEL_NAME,
            "sample_rows": CFG["shap_rows"],
            "features": ACTIVE_FEATURES,
            "test_only": True,
        }
        if stage_is_valid(
            "shap", [shap_path, shap_importance_path], shap_params
        ):
            loaded = np.load(shap_path, allow_pickle=True)
            SHAP_VALUES = loaded["shap_values"]
            SHAP_X = loaded["x_transformed"]
            SHAP_Y = loaded["y"]
            SHAP_FEATURE_NAMES = loaded["feature_names"].astype(str)
            shap_source_importance = pd.read_parquet(shap_importance_path)
            print("Loaded compatible SHAP checkpoint.")
        else:
            shap_pos = stratified_sample_positions(
                y_test, CFG["shap_rows"], RANDOM_STATE
            )
            shap_pipeline = TRAINED_MODELS[SHAP_MODEL_NAME]
            shap_preprocessor = shap_pipeline.named_steps["preprocess"]
            shap_estimator = shap_pipeline.named_steps["model"]
            transformed = shap_preprocessor.transform(X_test.iloc[shap_pos])
            if hasattr(transformed, "toarray"):
                transformed = transformed.toarray()
            transformed = np.asarray(transformed, dtype=np.float32)
            feature_names_transformed = np.asarray(
                shap_preprocessor.get_feature_names_out(), dtype=str
            )
            explainer = shap.TreeExplainer(shap_estimator)
            try:
                raw_shap_values = explainer.shap_values(
                    transformed, check_additivity=False
                )
            except TypeError:
                raw_shap_values = explainer.shap_values(transformed)
            SHAP_VALUES = normalize_multiclass_shap(
                raw_shap_values,
                n_samples=len(shap_pos),
                n_features=transformed.shape[1],
                n_classes=4,
            ).astype(np.float32)
            SHAP_X = transformed
            SHAP_Y = y_test.iloc[shap_pos].to_numpy()
            SHAP_FEATURE_NAMES = feature_names_transformed

            source_features = [
                transformed_to_raw_feature(name) for name in SHAP_FEATURE_NAMES
            ]
            importance_rows = []
            for source_feature in sorted(set(source_features)):
                cols = np.flatnonzero(np.asarray(source_features) == source_feature)
                for c in CLASS_ORDER:
                    importance_rows.append({
                        "source_feature": source_feature,
                        "class_code": c,
                        "class_label": CLASS_LABELS[c],
                        "mean_abs_shap": float(
                            np.abs(SHAP_VALUES[:, cols, c]).sum(axis=1).mean()
                        ),
                        "mean_signed_shap": float(
                            SHAP_VALUES[:, cols, c].sum(axis=1).mean()
                        ),
                    })
            shap_source_importance = pd.DataFrame(importance_rows)
            atomic_savez(
                shap_path,
                shap_values=SHAP_VALUES,
                x_transformed=SHAP_X,
                y=SHAP_Y,
                feature_names=SHAP_FEATURE_NAMES,
            )
            atomic_to_parquet(
                shap_source_importance, shap_importance_path, index=False
            )
            mark_stage_complete(
                "shap", [shap_path, shap_importance_path], shap_params
            )

        atomic_to_parquet(
            shap_source_importance.sort_values(["class_code", "mean_abs_shap"], ascending=[True, False]),
            DIRS["tables"] / "shap_class_specific_policy_table.parquet",
            index=False,
        )
        severe_importance = (
            shap_source_importance.loc[
                shap_source_importance["class_code"] == 3
            ]
            .sort_values("mean_abs_shap", ascending=False)
            .head(20)
            .sort_values("mean_abs_shap")
        )
        fig, ax = plt.subplots(figsize=(9, 7))
        ax.barh(
            severe_importance["source_feature"],
            severe_importance["mean_abs_shap"],
            color=CLASS_COLORS[3],
        )
        ax.set_xlabel("Mean absolute SHAP value (aggregated one-hot features)")
        ax.set_title(f"{SHAP_MODEL_NAME}: severe-class held-out SHAP importance")
        plt.tight_layout()
        plt.savefig(DIRS["figures"] / "09_shap_severe_source_features.png", bbox_inches="tight")
        plt.show()

        overall_source = (
            shap_source_importance.groupby("source_feature")["mean_abs_shap"]
            .mean().sort_values(ascending=False).head(15).index
        )
        heat = (
            shap_source_importance[
                shap_source_importance["source_feature"].isin(overall_source)
            ]
            .pivot(index="source_feature", columns="class_label", values="mean_abs_shap")
            .reindex(overall_source)
            .reindex(columns=[CLASS_LABELS[c] for c in CLASS_ORDER])
        )
        fig, ax = plt.subplots(figsize=(10, 7))
        sns.heatmap(heat, cmap="viridis", annot=True, fmt=".3g", ax=ax)
        ax.set_title("Class-specific held-out SHAP importance")
        ax.set_xlabel("Outcome class"); ax.set_ylabel("Source feature")
        plt.tight_layout()
        plt.savefig(DIRS["figures"] / "10_shap_multiclass_heatmap.png", bbox_inches="tight")
        plt.show()

        # A transformed-feature beeswarm for the severe class.
        shap.summary_plot(
            SHAP_VALUES[:, :, 3],
            SHAP_X,
            feature_names=SHAP_FEATURE_NAMES,
            max_display=20,
            show=False,
        )
        plt.title(f"{SHAP_MODEL_NAME}: severe-class SHAP beeswarm")
        plt.tight_layout()
        plt.savefig(DIRS["figures"] / "11_shap_severe_beeswarm.png", bbox_inches="tight")
        plt.show()
    else:
        print("SHAP skipped because no tree model completed.")


## 20. Optional NFHS-4 cross-wave transportability

Provide `EXTERNAL_NFHS4_PATH` to evaluate the frozen NFHS-5 model on harmonised NFHS-4 fields. This is a cross-wave transportability test, not prospective validation, because the development wave is later than the external wave.

No retraining, recalibration or category remapping may use external outcomes.


In [ ]:
# 20A. External/cross-wave validation
external_requirement_path = DIRS["reports"] / "external_validation_requirement.json"
if EXTERNAL_NFHS4_PATH:
    external_path = Path(EXTERNAL_NFHS4_PATH)
    if not external_path.exists():
        raise FileNotFoundError(external_path)
    external_hash = sha256_file(external_path, HASH_MODE)
    external_result_path = DIRS["tables"] / "nfhs4_cross_wave_metrics.json"
    external_pred_path = DIRS["predictions"] / "nfhs4_cross_wave_proba.npz"
    external_params = {
        "external_hash": external_hash,
        "primary_name": PRIMARY_NAME,
        "features": ACTIVE_FEATURES,
    }
    if stage_is_valid(
        "external_nfhs4",
        [external_result_path, external_pred_path],
        external_params,
    ):
        external_metrics = json.loads(external_result_path.read_text())
        print("Loaded compatible cross-wave validation checkpoint.")
    else:
        ext_raw, ext_labels = load_nfhs_source(external_path)
        if "row_id" not in ext_raw:
            ext_raw.insert(0, "row_id", np.arange(len(ext_raw), dtype=np.int64))
        ext = construct_analysis_frame(ext_raw)
        ext = ext.loc[ext["anaemia_level"].notna()].copy()
        ext_y = ext["anaemia_level"].astype(int)
        ext_X = ext.reindex(columns=ACTIVE_FEATURES)
        ext_w = ext["sample_weight"]
        ext_proba = normalize_probabilities(
            PRIMARY_PREDICTOR.predict_proba(ext_X)
        ).astype(np.float32)
        external_metrics = {
            "dataset": "NFHS-4 cross-wave transportability",
            "n": len(ext),
            "source_hash": external_hash,
            "warning": (
                "Development wave is later than evaluation wave; do not label "
                "this as prospective temporal validation."
            ),
            **metric_bundle(ext_y, ext_proba),
            **metric_bundle(
                ext_y, ext_proba, ext_w, prefix="survey_weighted_"
            ),
        }
        atomic_write_json(external_metrics, external_result_path)
        atomic_savez(external_pred_path, proba=ext_proba)
        mark_stage_complete(
            "external_nfhs4",
            [external_result_path, external_pred_path],
            external_params,
        )
    print(json.dumps(external_metrics, indent=2))
else:
    external_metrics = None
    requirement = {
        "status": "not_run",
        "requirement": (
            "Supply the raw NFHS-4 India IR file and set EXTERNAL_NFHS4_PATH. "
            "The frozen NFHS-5 predictor will be evaluated without retraining."
        ),
        "preferred_stronger_future_design": (
            "Develop on the earlier wave and validate on the later wave, or "
            "validate prospectively in an independent contemporary dataset."
        ),
    }
    atomic_write_json(requirement, external_requirement_path)
    print(requirement["requirement"])


## 21. Model export and safe research inference

The exported predictor contains fold-fitted preprocessing, trained base model(s)
and probability calibration. The helper accepts only raw predictor fields from
the final schema. It rejects haemoglobin and anaemia outcome fields, and does
not request respondent IDs, survey design fields or district identifiers.
Outputs are research probabilities, not clinical diagnoses.


In [ ]:
# 21A. Export schema and research-only inference helper
NON_PREDICTOR_RAW_FIELDS = {
    "v002", "v003", "v005", "v021", "v022", "sdist", "v456", "v457"
}
RAW_INFERENCE_FIELDS = [
    c for c in FINAL_RAW_COLUMNS if c not in NON_PREDICTOR_RAW_FIELDS
]
inference_schema = {
    "raw_input_fields": RAW_INFERENCE_FIELDS,
    "optional_at_inference_but_imputed": RAW_INFERENCE_FIELDS,
    "active_engineered_features": ACTIVE_FEATURES,
    "class_labels": CLASS_LABELS,
    "primary_model": PRIMARY_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "run_fingerprint": RUN_FINGERPRINT,
    "prohibited_inputs": sorted(OUTCOME_DERIVED_COLUMNS),
    "disclaimer": (
        "Research-only contemporaneous survey classification; not diagnosis, "
        "treatment guidance, prognosis or a validated clinical device."
    ),
}
atomic_write_json(inference_schema, DIRS["models"] / "inference_schema.json")


def predict_research_anaemia_risk(raw_records):
    if isinstance(raw_records, dict):
        raw_records = pd.DataFrame([raw_records])
    else:
        raw_records = pd.DataFrame(raw_records).copy()
    raw_records.columns = raw_records.columns.str.lower()
    prohibited_present = sorted(set(raw_records.columns) & OUTCOME_DERIVED_COLUMNS)
    if prohibited_present:
        raise ValueError(
            f"Outcome-derived inputs are prohibited at inference: {prohibited_present}"
        )
    unexpected = sorted(set(raw_records.columns) - set(RAW_INFERENCE_FIELDS))
    if unexpected:
        warnings.warn(f"Unexpected fields will be ignored: {unexpected}")
    raw_records = raw_records.reindex(columns=RAW_INFERENCE_FIELDS)
    raw_records.insert(0, "row_id", np.arange(len(raw_records), dtype=np.int64))
    raw_records["v457"] = np.nan  # builder placeholder; never used by the model
    raw_records["v456"] = np.nan
    engineered = construct_analysis_frame(raw_records)
    model_input = engineered.reindex(columns=ACTIVE_FEATURES)
    proba = normalize_probabilities(
        PRIMARY_PREDICTOR.predict_proba(model_input)
    )
    pred = proba.argmax(axis=1)
    output = pd.DataFrame({
        "predicted_class": pred,
        "predicted_label": [CLASS_LABELS[int(c)] for c in pred],
        **{
            f"probability_{CLASS_LABELS[c].lower().replace(' ', '_')}": proba[:, c]
            for c in CLASS_ORDER
        },
    })
    output["research_disclaimer"] = inference_schema["disclaimer"]
    return output


print("Primary model artifact:", primary_model_path)
print("Inference schema:", DIRS["models"] / "inference_schema.json")


## 22. Reproducibility report, model card and readiness gates

The final report distinguishes successful internal validation from missing
external validation. Publication readiness additionally requires the exact
40-column schema, target reconciliation, unique respondent identifiers,
survey-disjoint splits, completed stratified-PSU bootstrap and all configured
nested/geographic analyses. The report never upgrades internal validation into
a clinical or causal claim.


In [ ]:
# 22A. Honest readiness report, model card and methods summary
if FINAL_TEST_UNLOCKED and not locked_test_metrics.empty:
    primary_metrics = locked_test_metrics.loc[
        locked_test_metrics["candidate"] == PRIMARY_NAME
    ].iloc[0].to_dict()
else:
    primary_metrics = {}

readiness_checks = {
    "run_mode_is_paper": RUN_MODE == "paper",
    "exact_final_40_column_contract": contract_report["paper_contract_passed"],
    "expected_source_profile": all(contract_report["profile_checks"].values()),
    "correct_target_mapping": expected_mapping_check == [3, 2, 1, 0],
    "no_target_leakage": not bool(set(ACTIVE_FEATURES) & OUTCOME_DERIVED_COLUMNS),
    "unique_respondent_keys": analysis_df["respondent_key"].is_unique,
    "psu_disjoint_splits": (
        group_set(TRAIN_POS).isdisjoint(group_set(CALIB_POS))
        and group_set(TRAIN_POS).isdisjoint(group_set(TEST_POS))
        and group_set(CALIB_POS).isdisjoint(group_set(TEST_POS))
    ),
    "separate_calibration_set": len(CALIB_POS) > 0,
    "nested_cv_complete": (
        RUN_MODE == "paper" and len(nested_results) == CFG["nested_outer"]
    ),
    "geographic_validation_complete": (
        RUN_MODE == "paper" and len(geographic_results) == 5
    ),
    "final_test_explicitly_unlocked": FINAL_TEST_UNLOCKED,
    "locked_test_evaluated": (
        FINAL_TEST_UNLOCKED and len(PRIMARY_TEST_PROBA) == len(TEST_POS)
    ),
    "stratified_psu_bootstrap_500_complete": (
        FINAL_TEST_UNLOCKED and len(bootstrap_df) >= 500
    ),
    "probability_normalization_enforced": (
        PROBABILITY_POLICY_VERSION == "clip_renormalize_float64_v1"
    ),
    "pilot_test_exposure_documented": PILOT_TEST_EXPOSED,
    "external_validation_complete": external_metrics is not None,
}
internal_required = [
    "run_mode_is_paper",
    "exact_final_40_column_contract",
    "expected_source_profile",
    "correct_target_mapping",
    "no_target_leakage",
    "unique_respondent_keys",
    "psu_disjoint_splits",
    "separate_calibration_set",
    "nested_cv_complete",
    "geographic_validation_complete",
    "final_test_explicitly_unlocked",
    "locked_test_evaluated",
    "stratified_psu_bootstrap_500_complete",
    "probability_normalization_enforced",
    "pilot_test_exposure_documented",
]
internal_validation_complete = all(readiness_checks[key] for key in internal_required)
publication_ready = (
    internal_validation_complete and readiness_checks["external_validation_complete"]
)
if RUN_MODE != "paper":
    claim = "Development/pilot workflow only; not internally validated for publication."
elif not FINAL_TEST_UNLOCKED:
    claim = "Paper prerequisites may be complete, but the final test remains locked."
elif not internal_validation_complete:
    claim = "Paper-mode run incomplete; inspect failed readiness gates."
elif not publication_ready:
    claim = "Internal paper-mode evaluation complete; external validation remains required."
else:
    claim = (
        "Internal and cross-wave validation complete; manuscript reporting and "
        "risk-of-bias review remain required."
    )

readiness_report = {
    "checks": readiness_checks,
    "internal_validation_complete": internal_validation_complete,
    "publication_ready": publication_ready,
    "publication_claim": claim,
}
atomic_write_json(readiness_report, DIRS["reports"] / "readiness_report.json")

if primary_metrics:
    metric_lines = "\n".join([
        f"- Macro F1: {primary_metrics.get('macro_f1', float('nan')):.4f}",
        f"- Balanced accuracy: {primary_metrics.get('balanced_accuracy', float('nan')):.4f}",
        f"- Macro AUPRC: {primary_metrics.get('macro_auprc', float('nan')):.4f}",
        f"- Severe recall: {primary_metrics.get('recall_class_3', float('nan')):.4f}",
        f"- Severe AUPRC: {primary_metrics.get('auprc_class_3', float('nan')):.4f}",
        f"- Multiclass Brier score: {primary_metrics.get('brier', float('nan')):.4f}",
        f"- Expected calibration error: {primary_metrics.get('ece', float('nan')):.4f}",
    ])
else:
    metric_lines = "- Not reported: the final test remains locked."

model_card = f"""
# Model Card — NFHS-5 Four-Class Anaemia Research Classifier

## Identity
- Pipeline version: {PIPELINE_VERSION}
- Run fingerprint: {RUN_FINGERPRINT}
- Run mode: {RUN_MODE}
- Primary predictor: {PRIMARY_NAME}
- Final test unlocked: {FINAL_TEST_UNLOCKED}

## Intended use
Contemporaneous research classification, methodological comparison and
public-health subgroup analysis.

## Out-of-scope use
- Diagnosis, treatment decisions or future prognosis
- Causal interpretation of predictors or SHAP values
- Deployment in populations without external validation

## Outcome
0 No anaemia; 1 Mild; 2 Moderate; 3 Severe. Raw v457 codes 4, 3, 2 and 1
were mapped respectively.

## Evaluation status
{claim}

## Locked-test headline metrics
{metric_lines}

## Known limitations
- The v3 development test was exposed and is retained only as a pilot result.
- Cross-sectional associations are not temporal or causal inference.
- Severe anaemia is rare; class-specific estimates require cluster uncertainty.
- Social variables may encode structural inequities.
- Food-frequency variables are coarse and not quantitative dietary intake.
- The v456/v457 comparison is a coding audit, not independent validation.
- Cost matrices are stakeholder scenarios, not WHO-prescribed costs.
- External/prospective validation is required before applied use.
""".strip()
(DIRS["reports"] / "model_card.md").write_text(model_card, encoding="utf-8")

test_sentence = (
    "The explicitly unlocked final test was evaluated after nested and geographic "
    "validation gates completed."
    if FINAL_TEST_UNLOCKED
    else "The final test remained locked and no final-test metrics were produced."
)
methods_summary = f"""
# Reproducible Methods Summary

The analysis used the validated NFHS-5 final 40-column research extract. The
four-class outcome mapped v457 codes 4→0, 3→1, 2→2 and 1→3. Haemoglobin-derived
variables and survey identifiers were excluded from predictors. Survey weights
supported population descriptions and weighted evaluation. PSUs were disjoint
across training, calibration and test sets. Preprocessing and missingness
indicators were fitted within grouped folds. Hyperparameters were selected with
persistent Optuna studies. {test_sentence} Confidence intervals resampled PSUs
within observed strata and use survey-weighted point estimates. SHAP is allowed
only after final-test unlock and is interpreted associationally.

The v3 development test exposure is documented in protocol_amendment_v31.json.
External/cross-wave validation remains a publication gate.

Run fingerprint: {RUN_FINGERPRINT}
""".strip()
(DIRS["reports"] / "methods_summary.md").write_text(
    methods_summary, encoding="utf-8"
)

print(json.dumps(readiness_report, indent=2))
print("Model card:", DIRS["reports"] / "model_card.md")
print("Methods summary:", DIRS["reports"] / "methods_summary.md")


In [ ]:
# 22B. Final artifact inventory
artifact_inventory = []
for artifact_type, directory in DIRS.items():
    for path in sorted(directory.glob("*")):
        if path.is_file():
            artifact_inventory.append({
                "artifact_type": artifact_type,
                "name": path.name,
                "path": str(path),
                "size_bytes": path.stat().st_size,
            })
artifact_inventory = pd.DataFrame(artifact_inventory)
atomic_to_parquet(
    artifact_inventory,
    DIRS["reports"] / "artifact_inventory.parquet",
    index=False,
)

print("=" * 88)
print(
    "PAPER PIPELINE STATUS" if RUN_MODE == "paper"
    else "DEVELOPMENT/PILOT PIPELINE STATUS"
)
print("=" * 88)
print("Run fingerprint :", RUN_FINGERPRINT)
print("Primary model   :", PRIMARY_NAME)
print("Rows            :", f"{len(analysis_df):,}")
print("Raw columns     :", len(FINAL_RAW_COLUMNS))
print("Model features  :", len(ACTIVE_FEATURES))
print("Train/Cal/Test  :", len(TRAIN_POS), len(CALIB_POS), len(TEST_POS))
print("Test unlocked   :", FINAL_TEST_UNLOCKED)
print("Checkpoint root :", RUN_DIR)
print("Artifacts       :", len(artifact_inventory))
print("Readiness       :", readiness_report["publication_claim"])
print("=" * 88)


## References and reporting resources

1. International Institute for Population Sciences (IIPS) and ICF. *National
   Family Health Survey (NFHS-5), 2019–21: India*. Mumbai: IIPS.
2. The DHS Program. *DHS-7 Standard Recode Manual*.
3. World Health Organization. *Guideline on haemoglobin cutoffs to define
   anaemia in individuals and populations* (2024).
4. Collins GS, et al. TRIPOD+AI statement for reporting clinical prediction
   models using regression or machine learning methods.
5. Wolff RF, et al. PROBAST: a tool to assess risk of bias and applicability of
   prediction model studies.
6. Lundberg SM, Lee S-I. A unified approach to interpreting model predictions.

### Required manuscript disclosures

- NFHS data-access permission and citation
- Cross-sectional, contemporaneous prediction design
- Exact 40-column raw data contract and target mapping
- Survey weights, PSU/strata construction and state/district coverage
- Missing/special-code handling and feature definitions
- Complete train/calibration/test and nested-CV design
- All hyperparameter search spaces and random seeds
- Calibration, bootstrap, subgroup and state-held-out results
- WHO sensitivity limitation caused by absent pregnancy duration
- No causal, diagnostic or prospective-performance claim
